<a href="https://colab.research.google.com/github/AnjanPayra/Adam_DeepLearning/blob/main/DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import random
import pickle

In [3]:
# ================================================================
# CELL 1: IMPORTS + COMMON CONFIGURATION
# Multi-network pipeline:
# YDIP | YMIPS | YMBD | YHQ
#
# Compatible with:
#     Step 1 - Feature Engineering
#     Step 2 - Basic MLP
#     Step 3 - Advanced Deep NN
#     Step 4 - Existing pipeline step
#     Step 5 - Adam DNN + Quantum VQC
#     Step 6 - Full Validation & Comparison
#     run_full_pipeline()
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import sys
import numpy as np
import pandas as pd
import networkx as nx
import joblib
import pickle
import itertools
import random
import json

from collections import defaultdict


# ================================================================
# 2. SCIKIT-LEARN IMPORTS
# ================================================================

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef
)


# ================================================================
# 3. MATPLOTLIB
# ================================================================

import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt


# ================================================================
# 4. RANDOM SEED
# ================================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

random.seed(
    RANDOM_STATE
)


# ================================================================
# 5. WORKING DIRECTORY
# ================================================================

WORK_DIR = '/content'

if WORK_DIR not in sys.path:

    sys.path.insert(
        0,
        WORK_DIR
    )


# ================================================================
# 6. IMPORT CUSTOM MODEL CLASSES
#
# Required files:
#
#     /content/deep_nn.py
#     /content/adam_plain_nn.py
#     /content/quantum_vqc.py
#
# ================================================================

from deep_nn import DeepNN

from adam_plain_nn import AdamPlainNN

from quantum_vqc import VQC


# ================================================================
# 7. COMMON INPUT FILES
#
# These files are shared by all four networks.
# ================================================================

COMPLEX_FILE = (

    '/content/complex_network.xlsx'

)


ESSENTIAL_FILE = (

    '/content/Essential.xlsx'

)


# ================================================================
# 8. NETWORK NAMES
#
# IMPORTANT:
#
# This was missing in your original Cell 1.
#
# Your code later used:
#
#     for network_name in NETWORKS
#
# Therefore NETWORKS must be explicitly defined.
# ================================================================

NETWORKS = [

    'YDIP',

    'YMIPS',

    'YMBD',

    'YHQ'

]


# ================================================================
# 9. DEFINE NETWORK-SPECIFIC CONFIGURATION
# ================================================================

NETWORK_CONFIGS = {

    'YDIP': {

        'ppi_file':
            '/content/YDIP.csv',

        'complex_file':
            COMPLEX_FILE,

        'essential_file':
            ESSENTIAL_FILE

    },


    'YMIPS': {

        'ppi_file':
            '/content/YMIPS.csv',

        'complex_file':
            COMPLEX_FILE,

        'essential_file':
            ESSENTIAL_FILE

    },


    'YMBD': {

        'ppi_file':
            '/content/YMBD.csv',

        'complex_file':
            COMPLEX_FILE,

        'essential_file':
            ESSENTIAL_FILE

    },


    'YHQ': {

        'ppi_file':
            '/content/YHQ.csv',

        'complex_file':
            COMPLEX_FILE,

        'essential_file':
            ESSENTIAL_FILE

    }

}


# ================================================================
# 10. VERIFY NETWORK CONFIGURATION
# ================================================================

for network_name in NETWORKS:

    if network_name not in NETWORK_CONFIGS:

        raise KeyError(

            f"Network '{network_name}' is missing "
            f"from NETWORK_CONFIGS."

        )


# ================================================================
# 11. CREATE NETWORK-SPECIFIC PATHS
# ================================================================

def create_paths(
    network_name
):

    """
    Create all paths required for one network.

    The returned dictionary is intentionally FLAT.

    This is important because Steps 1-6 expect:

        PATHS[network_name]['features_file']

        PATHS[network_name]['splits_file']

        PATHS[network_name]['scaler_file']

        PATHS[network_name]['basic_model_file']

        PATHS[network_name]['advanced_model_file']

        PATHS[network_name]['adam_model_file']

        PATHS[network_name]['quantum_model_file']

    """

    # ------------------------------------------------------------
    # Network output directory
    # ------------------------------------------------------------

    output_dir = os.path.join(

        WORK_DIR,

        f'{network_name}_results'

    )


    os.makedirs(

        output_dir,

        exist_ok=True

    )


    # ------------------------------------------------------------
    # Create FLAT path dictionary
    # ------------------------------------------------------------

    paths = {


        # ========================================================
        # GENERAL
        # ========================================================

        'output_dir':

            output_dir,


        # ========================================================
        # STEP 1
        # Feature Engineering
        # ========================================================

        'features_file':

            os.path.join(

                output_dir,

                'protein_features.csv'

            ),


        # ========================================================
        # STEP 2
        # Basic MLP
        # ========================================================

        'splits_file':

            os.path.join(

                output_dir,

                'splits.npz'

            ),


        'scaler_file':

            os.path.join(

                output_dir,

                'scaler.joblib'

            ),


        'basic_model_file':

            os.path.join(

                output_dir,

                'basic_model.joblib'

            ),


        'basic_search_file':

            os.path.join(

                output_dir,

                'basic_search_results.csv'

            ),


        # ========================================================
        # STEP 3
        # Advanced Deep NN
        # ========================================================

        'advanced_model_file':

            os.path.join(

                output_dir,

                'advanced_model.pkl'

            ),


        'advanced_search_file':

            os.path.join(

                output_dir,

                'hyperparam_search_results.csv'

            ),


        # ========================================================
        # STEP 5A
        # Adam-Optimized DNN
        # ========================================================

        'adam_model_file':

            os.path.join(

                output_dir,

                'adam_plain_model.pkl'

            ),


        'adam_search_file':

            os.path.join(

                output_dir,

                'adam_search_results.csv'

            ),


        # ========================================================
        # STEP 5B
        # Quantum VQC
        # ========================================================

        'quantum_model_file':

            os.path.join(

                output_dir,

                'quantum_vqc_model.pkl'

            ),


        'quantum_search_file':

            os.path.join(

                output_dir,

                'quantum_search_results.csv'

            ),


        # ========================================================
        # STEP 6
        # Four-model comparison
        # ========================================================

        'comparison_plot':

            os.path.join(

                output_dir,

                'model_comparison_4models.png'

            ),


        'comparison_table':

            os.path.join(

                output_dir,

                'final_comparison_table_4models.csv'

            )

    }


    return paths


# ================================================================
# 12. CREATE PATH CONFIGURATION FOR ALL NETWORKS
#
# IMPORTANT:
#
# PATHS is the variable expected by your existing:
#
#     Step 1
#     Step 2
#     Step 3
#     Step 5
#     Step 6
#     run_full_pipeline()
#
# ================================================================

PATHS = {}


for network_name in NETWORKS:

    PATHS[
        network_name
    ] = create_paths(

        network_name

    )


# ================================================================
# 13. OPTIONAL COMPATIBILITY ALIAS
#
# Some earlier versions of your code used NETWORK_PATHS.
#
# We keep NETWORK_PATHS available so older code does not break.
#
# ================================================================

NETWORK_PATHS = {}


for network_name in NETWORKS:

    NETWORK_PATHS[
        network_name
    ] = {

        'output_dir':

            PATHS[
                network_name
            ][
                'output_dir'
            ],

        'paths':

            PATHS[
                network_name
            ]

    }


# ================================================================
# 14. DISPLAY COMPLETE CONFIGURATION
# ================================================================

print(

    '=' * 80

)

print(

    'MULTI-NETWORK PIPELINE CONFIGURATION'

)

print(

    '=' * 80
)


for network_name in NETWORKS:

    config = NETWORK_CONFIGS[
        network_name
    ]


    print(

        f'\nNetwork: {network_name}'

    )


    print(

        f'  PPI file        : '
        f'{config["ppi_file"]}'

    )


    print(

        f'  Complex file    : '
        f'{config["complex_file"]}'

    )


    print(

        f'  Essential labels: '
        f'{config["essential_file"]}'

    )


    print(

        f'  Output directory: '
        f'{PATHS[network_name]["output_dir"]}'

    )


# ================================================================
# 15. VERIFY REQUIRED INPUT FILES
# ================================================================

print(

    '\n' + '=' * 80

)

print(

    'VERIFYING INPUT FILES'

)

print(

    '=' * 80
)


all_input_files_exist = True


for network_name in NETWORKS:

    config = NETWORK_CONFIGS[
        network_name
    ]


    ppi_file = config[
        'ppi_file'
    ]


    if os.path.exists(
        ppi_file
    ):

        print(

            f'[OK] {network_name} PPI file found: '
            f'{ppi_file}'

        )

    else:

        print(

            f'[MISSING] {network_name} PPI file NOT found: '
            f'{ppi_file}'

        )

        all_input_files_exist = False


# ================================================================
# 16. VERIFY COMMON COMPLEX FILE
# ================================================================

if os.path.exists(

    COMPLEX_FILE

):

    print(

        f'[OK] Complex file found: '
        f'{COMPLEX_FILE}'

    )

else:

    print(

        f'[MISSING] Complex file NOT found: '
        f'{COMPLEX_FILE}'

    )

    all_input_files_exist = False


# ================================================================
# 17. VERIFY COMMON ESSENTIAL-LABEL FILE
# ================================================================

if os.path.exists(

    ESSENTIAL_FILE

):

    print(

        f'[OK] Essential-label file found: '
        f'{ESSENTIAL_FILE}'

    )

else:

    print(

        f'[MISSING] Essential-label file NOT found: '
        f'{ESSENTIAL_FILE}'

    )

    all_input_files_exist = False


# ================================================================
# 18. VERIFY CUSTOM MODEL MODULES
# ================================================================

print(

    '\n' + '=' * 80

)

print(

    'VERIFYING CUSTOM MODEL MODULES'

)

print(

    '=' * 80
)


custom_modules = [

    'deep_nn.py',

    'adam_plain_nn.py',

    'quantum_vqc.py'

]


all_modules_exist = True


for module_file in custom_modules:

    module_path = os.path.join(

        WORK_DIR,

        module_file

    )


    if os.path.exists(

        module_path

    ):

        print(

            f'[OK] Custom module found: '
            f'{module_path}'

        )

    else:

        print(

            f'[MISSING] Custom module NOT found: '
            f'{module_path}'

        )

        all_modules_exist = False


# ================================================================
# 19. FINAL CONFIGURATION CHECK
# ================================================================

print(

    '\n' + '=' * 80

)


if (

    all_input_files_exist

    and

    all_modules_exist

):

    print(

        'SUCCESS: All required input files and '
        'custom model modules are available.'

    )

else:

    print(

        'WARNING: One or more required files are missing.'

    )

    print(

        'Please upload the missing files to /content '
        'before running the pipeline.'

    )


# ================================================================
# 20. PRINT EXPECTED OUTPUT STRUCTURE
# ================================================================

print(

    '\n' + '=' * 80

)

print(

    'EXPECTED OUTPUT DIRECTORIES'

)

print(

    '=' * 80
)


for network_name in NETWORKS:

    print(

        f'\n{network_name}_results/'

    )


    print(

        '  ├── protein_features.csv'

    )


    print(

        '  ├── splits.npz'

    )


    print(

        '  ├── scaler.joblib'

    )


    print(

        '  ├── basic_model.joblib'

    )


    print(

        '  ├── basic_search_results.csv'

    )


    print(

        '  ├── advanced_model.pkl'

    )


    print(

        '  ├── hyperparam_search_results.csv'

    )


    print(

        '  ├── adam_plain_model.pkl'

    )


    print(

        '  ├── adam_search_results.csv'

    )


    print(

        '  ├── quantum_vqc_model.pkl'

    )


    print(

        '  ├── quantum_search_results.csv'

    )


    print(

        '  ├── model_comparison_4models.png'

    )


    print(

        '  └── final_comparison_table_4models.csv'

    )


# ================================================================
# 21. FINAL VARIABLE COMPATIBILITY CHECK
# ================================================================

print(

    '\n' + '=' * 80

)

print(

    'CHECKING PIPELINE VARIABLES'

)

print(

    '=' * 80
)


print(

    f'NETWORKS defined        : {NETWORKS}'

)


print(

    f'NETWORK_CONFIGS defined : '
    f'{list(NETWORK_CONFIGS.keys())}'

)


print(

    f'PATHS defined           : '
    f'{list(PATHS.keys())}'

)


print(

    f'NETWORK_PATHS defined   : '
    f'{list(NETWORK_PATHS.keys())}'

)


# ================================================================
# 22. CELL 1 COMPLETE
# ================================================================

print(

    '\n' + '=' * 80

)

print(

    'CELL 1 COMPLETE'

)

print(

    '=' * 80

)


print(

    '\nYou can now run the pipeline cells in order:'

)


print(

    '1. Step 1 - Feature Engineering'

)


print(

    '2. Step 2 - Basic MLP'


)


print(

    '3. Step 3 - Advanced Deep NN'


)


print(

    '4. Step 4 - Existing model step'


)


print(

    '5. Step 5 - Adam DNN + Quantum VQC'


)


print(

    '6. Step 6 - Full Validation & Comparison'


)


print(

    '7. run_full_pipeline()'


)


print(

    '\nThe corrected variable for network paths is now: PATHS'

)


print(

    'Example:'

)


print(

    "PATHS['YDIP']['splits_file']"

)


print(

    "PATHS['YMIPS']['adam_model_file']"

)


print(

    "PATHS['YHQ']['quantum_model_file']"

)


print(

    '\n' + '=' * 80

)


MULTI-NETWORK PIPELINE CONFIGURATION

Network: YDIP
  PPI file        : /content/YDIP.csv
  Complex file    : /content/complex_network.xlsx
  Essential labels: /content/Essential.xlsx
  Output directory: /content/YDIP_results

Network: YMIPS
  PPI file        : /content/YMIPS.csv
  Complex file    : /content/complex_network.xlsx
  Essential labels: /content/Essential.xlsx
  Output directory: /content/YMIPS_results

Network: YMBD
  PPI file        : /content/YMBD.csv
  Complex file    : /content/complex_network.xlsx
  Essential labels: /content/Essential.xlsx
  Output directory: /content/YMBD_results

Network: YHQ
  PPI file        : /content/YHQ.csv
  Complex file    : /content/complex_network.xlsx
  Essential labels: /content/Essential.xlsx
  Output directory: /content/YHQ_results

VERIFYING INPUT FILES
[OK] YDIP PPI file found: /content/YDIP.csv
[OK] YMIPS PPI file found: /content/YMIPS.csv
[OK] YMBD PPI file found: /content/YMBD.csv
[OK] YHQ PPI file found: /content/YHQ.csv
[OK] Com

In [4]:
# ================================================================
# CELL 2: STEP 1 - FEATURE ENGINEERING FUNCTION
# ================================================================
#
# Automatically runs Step 1 for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# It uses the network-specific configuration created in CELL 1.
#
# Expected CELL 1 variables:
#
#     NETWORK_CONFIGS
#     PATHS
#
# Or, if your CELL 1 uses different variable names, replace them
# in the final execution cell accordingly.
#
# Outputs:
#
#     /content/YDIP_results/protein_features.csv
#     /content/YMIPS_results/protein_features.csv
#     /content/YMBD_results/protein_features.csv
#     /content/YHQ_results/protein_features.csv
#
# ================================================================


def run_step1_feature_engineering(
    network_name,
    config,
    paths
):

    # ============================================================
    # 1. GET NETWORK-SPECIFIC FILE PATHS
    # ============================================================

    PPI_FILE = config['ppi_file']
    COMPLEX_FILE = config['complex_file']
    ESSENTIAL_FILE = config['essential_file']

    OUTPUT_DIR = paths['output_dir']

    # IMPORTANT:
    # paths is already the current network's path dictionary.
    # Therefore DO NOT use paths['paths']['features_file'].
    FEATURES_FILE = paths['features_file']


    # ============================================================
    # 2. CREATE OUTPUT DIRECTORY
    # ============================================================

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ============================================================
    # 3. RANDOM STATE
    # ============================================================

    RANDOM_STATE = 42


    # ============================================================
    # 4. PRINT NETWORK INFORMATION
    # ============================================================

    print("\n")
    print("=" * 80)
    print(
        f"STEP 1: FEATURE ENGINEERING - {network_name}"
    )
    print("=" * 80)

    print(
        f"\nNetwork name   : {network_name}"
    )

    print(
        f"PPI file       : {PPI_FILE}"
    )

    print(
        f"Complex file   : {COMPLEX_FILE}"
    )

    print(
        f"Essential file : {ESSENTIAL_FILE}"
    )

    print(
        f"Output folder  : {OUTPUT_DIR}"
    )


    # ============================================================
    # 5. LOAD PPI NETWORK
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "1. Loading PPI network"
    )
    print("-" * 70)

    edges = pd.read_csv(
        PPI_FILE,
        header=None,
        names=[
            'p1',
            'p2'
        ]
    )


    # Remove missing protein identifiers

    edges = edges.dropna(
        subset=[
            'p1',
            'p2'
        ]
    ).copy()


    # Convert protein identifiers to strings

    edges['p1'] = (
        edges['p1']
        .astype(str)
        .str.strip()
    )

    edges['p2'] = (
        edges['p2']
        .astype(str)
        .str.strip()
    )


    # Remove empty identifiers

    edges = edges[
        (edges['p1'] != '') &
        (edges['p2'] != '')
    ].copy()


    # ============================================================
    # 6. BUILD PPI GRAPH
    # ============================================================

    G = nx.from_pandas_edgelist(
        edges,
        source='p1',
        target='p2'
    )


    # Remove self-loops

    G.remove_edges_from(
        nx.selfloop_edges(G)
    )


    print(
        f"PPI network: "
        f"{G.number_of_nodes()} proteins, "
        f"{G.number_of_edges()} interactions"
    )


    # ============================================================
    # 7. CHECK THAT PPI GRAPH IS NOT EMPTY
    # ============================================================

    if G.number_of_nodes() == 0:

        raise ValueError(
            f"{network_name}: "
            f"PPI graph contains no valid proteins."
        )


    # ============================================================
    # 8. LOAD PROTEIN-COMPLEX MEMBERSHIP NETWORK
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "2. Loading protein-complex membership network"
    )
    print("-" * 70)

    cx = pd.read_excel(
        COMPLEX_FILE,
        sheet_name='Sheet1',
        header=None
    )


    # ============================================================
    # 9. EXTRACT COMPLEX MEMBERS
    # ============================================================

    complexes = []


    for _, row in cx.iterrows():

        members = [
            str(x).strip()
            for x in row.tolist()
            if pd.notna(x)
        ]


        members = [
            protein
            for protein in members
            if protein != ''
        ]


        if members:

            complexes.append(
                members
            )


    print(
        f"Complex network: "
        f"{len(complexes)} complexes"
    )


    # ============================================================
    # 10. BUILD PROTEIN -> COMPLEX MAPPING
    # ============================================================

    prot2complex = defaultdict(list)


    for i, complex_members in enumerate(
        complexes
    ):

        for protein in complex_members:

            prot2complex[
                protein
            ].append(i)


    # ============================================================
    # 11. CALCULATE COMPLEX SIZES
    # ============================================================

    complex_sizes = np.array(
        [
            len(complex_members)
            for complex_members in complexes
        ]
    )


    print(
        f"Proteins with complex annotations: "
        f"{len(prot2complex)}"
    )


    # ============================================================
    # 12. BUILD CO-COMPLEX GRAPH
    # ============================================================

    print(
        "\nBuilding co-complex graph ..."
    )

    CoG = nx.Graph()


    # Add all PPI proteins

    CoG.add_nodes_from(
        G.nodes()
    )


    # Add edges between proteins sharing a complex

    for complex_members in complexes:

        for i in range(
            len(complex_members)
        ):

            for j in range(
                i + 1,
                len(complex_members)
            ):

                CoG.add_edge(
                    complex_members[i],
                    complex_members[j]
                )


    print(
        f"Co-complex network: "
        f"{CoG.number_of_nodes()} proteins, "
        f"{CoG.number_of_edges()} "
        f"co-complex interactions"
    )


    # ============================================================
    # 13. COMPUTE PPI TOPOLOGICAL FEATURES
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "3. Computing PPI topological features ..."
    )
    print("-" * 70)


    # Degree

    print(
        "Computing degree ..."
    )

    degree = dict(
        G.degree()
    )


    # Clustering coefficient

    print(
        "Computing clustering coefficient ..."
    )

    clustering = nx.clustering(
        G
    )


    # Betweenness centrality

    print(
        "Computing betweenness centrality ..."
    )

    betweenness_k = min(
        500,
        G.number_of_nodes()
    )


    if betweenness_k < G.number_of_nodes():

        betweenness = nx.betweenness_centrality(
            G,
            k=betweenness_k,
            seed=RANDOM_STATE
        )

    else:

        betweenness = nx.betweenness_centrality(
            G
        )


    # Closeness centrality

    print(
        "Computing closeness centrality ..."
    )

    closeness = nx.closeness_centrality(
        G
    )


    # PageRank

    print(
        "Computing PageRank ..."
    )

    pagerank = nx.pagerank(
        G,
        alpha=0.85
    )


    # Eigenvector centrality

    print(
        "Computing eigenvector centrality ..."
    )

    try:

        eigen = nx.eigenvector_centrality(
            G,
            max_iter=1000
        )

    except nx.PowerIterationFailedConvergence:

        print(
            "WARNING: Eigenvector centrality "
            "did not converge."
        )

        print(
            "Using zero values for eigenvector centrality."
        )

        eigen = {
            protein: 0.0
            for protein in G.nodes()
        }


    # Average neighbor degree

    print(
        "Computing average neighbor degree ..."
    )

    avg_nbr_deg = nx.average_neighbor_degree(
        G
    )


    # K-core number

    print(
        "Computing k-core number ..."
    )

    core_number = nx.core_number(
        G
    )


    # ============================================================
    # 14. COMPUTE COMPLEX-MEMBERSHIP FEATURES
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "4. Computing complex-membership features ..."
    )
    print("-" * 70)


    # Number of complexes containing a protein

    def n_complexes(protein):

        return len(
            prot2complex.get(
                protein,
                []
            )
        )


    # Mean complex size

    def mean_complex_size(protein):

        idxs = prot2complex.get(
            protein,
            []
        )

        if idxs:

            return float(
                np.mean(
                    [
                        len(complexes[i])
                        for i in idxs
                    ]
                )
            )

        return 0.0


    # Maximum complex size

    def max_complex_size(protein):

        idxs = prot2complex.get(
            protein,
            []
        )

        if idxs:

            return float(
                np.max(
                    [
                        len(complexes[i])
                        for i in idxs
                    ]
                )
            )

        return 0.0


    # Co-complex degree

    co_degree = dict(
        CoG.degree()
    )


    # Co-complex clustering coefficient

    co_clustering = nx.clustering(
        CoG
    )


    # ============================================================
    # 15. ASSEMBLE FEATURE TABLE
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "5. Assembling feature table ..."
    )
    print("-" * 70)


    rows = []


    for protein in G.nodes():

        rows.append(
            {

                'protein':
                    protein,

                # PPI topology

                'degree':
                    degree.get(
                        protein,
                        0
                    ),

                'clustering_coeff':
                    clustering.get(
                        protein,
                        0.0
                    ),

                'betweenness':
                    betweenness.get(
                        protein,
                        0.0
                    ),

                'closeness':
                    closeness.get(
                        protein,
                        0.0
                    ),

                'pagerank':
                    pagerank.get(
                        protein,
                        0.0
                    ),

                'eigenvector_centrality':
                    eigen.get(
                        protein,
                        0.0
                    ),

                'avg_neighbor_degree':
                    avg_nbr_deg.get(
                        protein,
                        0.0
                    ),

                'k_core':
                    core_number.get(
                        protein,
                        0
                    ),

                # Complex membership

                'n_complexes':
                    n_complexes(
                        protein
                    ),

                'mean_complex_size':
                    mean_complex_size(
                        protein
                    ),

                'max_complex_size':
                    max_complex_size(
                        protein
                    ),

                'co_complex_degree':
                    co_degree.get(
                        protein,
                        0
                    ),

                'co_complex_clustering':
                    co_clustering.get(
                        protein,
                        0.0
                    )
            }
        )


    feat_df = pd.DataFrame(
        rows
    )


    print(
        f"Feature table created: "
        f"{feat_df.shape[0]} proteins x "
        f"{feat_df.shape[1]} columns"
    )


    # ============================================================
    # 16. LOAD GROUND-TRUTH LABELS
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "6. Loading essential / non-essential "
        "ground-truth labels ..."
    )
    print("-" * 70)


    # Essential proteins

    ess = (
        pd.read_excel(
            ESSENTIAL_FILE,
            sheet_name='essential proteins',
            header=None
        )[0]
        .astype(str)
        .str.strip()
    )


    # Non-essential proteins

    noness = (
        pd.read_excel(
            ESSENTIAL_FILE,
            sheet_name='non-essential proteins',
            header=None
        )[0]
        .astype(str)
        .str.strip()
    )


    print(
        f"Essential proteins in ground truth: "
        f"{len(ess)}"
    )

    print(
        f"Non-essential proteins in ground truth: "
        f"{len(noness)}"
    )


    # ============================================================
    # 17. CREATE LABEL MAP
    # ============================================================

    label_map = {
        protein: 1
        for protein in ess
    }


    label_map.update(
        {
            protein: 0
            for protein in noness
        }
    )


    # ============================================================
    # 18. ASSIGN LABELS
    # ============================================================

    feat_df['label'] = (
        feat_df['protein']
        .map(label_map)
    )


    # ============================================================
    # 19. KEEP ONLY LABELED PROTEINS
    # ============================================================

    data = (
        feat_df
        .dropna(
            subset=[
                'label'
            ]
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    data['label'] = (
        data['label']
        .astype(int)
    )


    # ============================================================
    # 20. FINAL DATASET INFORMATION
    # ============================================================

    print("\n" + "-" * 70)
    print(
        "7. Final labeled dataset"
    )
    print("-" * 70)


    print(
        f"Final labeled dataset: "
        f"{data.shape[0]} proteins x "
        f"{data.shape[1] - 2} features"
    )


    print(
        "\nClass distribution:"
    )


    print(
        data['label']
        .value_counts()
        .rename(
            {
                1: 'essential',
                0: 'non-essential'
            }
        )
    )


    # ============================================================
    # 21. SAVE FEATURE DATASET
    # ============================================================

    data.to_csv(
        FEATURES_FILE,
        index=False
    )


    print(
        f"\nSaved feature dataset -> "
        f"{FEATURES_FILE}"
    )


    # ============================================================
    # 22. DISPLAY FIRST FIVE ROWS
    # ============================================================

    print(
        "\nFirst 5 rows:"
    )

    print(
        data.head()
    )


    # ============================================================
    # 23. STEP 1 COMPLETE
    # ============================================================

    print("\n" + "=" * 80)

    print(
        f"STEP 1 COMPLETED SUCCESSFULLY FOR "
        f"{network_name}"
    )

    print(
        "=" * 80
    )


    # ============================================================
    # 24. RETURN RESULTS
    # ============================================================

    return {
        'network_name':
            network_name,

        'data':
            data,

        'feature_file':
            FEATURES_FILE,

        'ppi_graph':
            G,

        'co_complex_graph':
            CoG,

        'n_proteins':
            G.number_of_nodes(),

        'n_interactions':
            G.number_of_edges(),

        'n_labeled_proteins':
            data.shape[0]
    }


# ================================================================
# CELL 3: RUN STEP 1 FOR ALL FOUR NETWORKS
# ================================================================
#
# This cell automatically runs the function for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# No manual switching of network names is required.
#
# The results are stored in:
#
#     STEP1_RESULTS['YDIP']
#     STEP1_RESULTS['YMIPS']
#     STEP1_RESULTS['YMBD']
#     STEP1_RESULTS['YHQ']
#
# ================================================================


STEP1_RESULTS = {}


for network_name in NETWORK_CONFIGS.keys():

    print("\n\n")

    print(
        "#" * 100
    )

    print(
        f"STARTING STEP 1 FOR NETWORK: "
        f"{network_name}"
    )

    print(
        "#" * 100
    )


    STEP1_RESULTS[
        network_name
    ] = run_step1_feature_engineering(
        network_name=network_name,
        config=NETWORK_CONFIGS[
            network_name
        ],
        paths=PATHS[
            network_name
        ]
    )


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n\n")

print(
    "=" * 100
)

print(
    "STEP 1 COMPLETED FOR ALL NETWORKS"
)

print(
    "=" * 100
)


for network_name, result in STEP1_RESULTS.items():

    print(
        f"\n{network_name}:"
    )

    print(
        f"  Proteins in PPI: "
        f"{result['n_proteins']}"
    )

    print(
        f"  PPI interactions: "
        f"{result['n_interactions']}"
    )

    print(
        f"  Labeled proteins: "
        f"{result['n_labeled_proteins']}"
    )

    print(
        f"  Feature file: "
        f"{result['feature_file']}"
    )




####################################################################################################
STARTING STEP 1 FOR NETWORK: YDIP
####################################################################################################


STEP 1: FEATURE ENGINEERING - YDIP

Network name   : YDIP
PPI file       : /content/YDIP.csv
Complex file   : /content/complex_network.xlsx
Essential file : /content/Essential.xlsx
Output folder  : /content/YDIP_results

----------------------------------------------------------------------
1. Loading PPI network
----------------------------------------------------------------------
PPI network: 5095 proteins, 24744 interactions

----------------------------------------------------------------------
2. Loading protein-complex membership network
----------------------------------------------------------------------
Complex network: 745 complexes
Proteins with complex annotations: 2167

Building co-complex graph ...
Co-complex network: 5286 proteins, 

In [5]:
# ================================================================
# CELL 3: STEP 2 - BASIC DEEP LEARNING MODEL
# ================================================================
#
# Trains a single-hidden-layer Multi-Layer Perceptron (MLP)
# classifier for the CURRENT PPI network.
#
# This function is automatically called for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# It uses the network-specific configuration from CELL 1.
#
# The same train / validation / test split is saved and reused
# by Step 3, Step 4, Step 5 and Step 6.
#
# Outputs:
#
#     {NETWORK}_results/scaler.joblib
#     {NETWORK}_results/basic_model.joblib
#     {NETWORK}_results/splits.npz
#
# ================================================================


def run_step2_basic_model(
    network_name,
    config,
    paths,
    step1_result=None
):

    # ============================================================
    # 1. GET NETWORK-SPECIFIC FILE PATHS
    # ============================================================

    FEATURES_FILE = paths['features_file']
    OUTPUT_DIR = paths['output_dir']

    SCALER_FILE = paths['scaler_file']
    BASIC_MODEL_FILE = paths['basic_model_file']
    SPLITS_FILE = paths['splits_file']


    # ============================================================
    # 2. RANDOM STATE
    # ============================================================

    RANDOM_STATE = 42


    # ============================================================
    # 3. CREATE OUTPUT DIRECTORY
    # ============================================================

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ============================================================
    # 4. PRINT NETWORK INFORMATION
    # ============================================================

    print("\n")
    print("=" * 80)
    print(
        f"STEP 2: BASIC DEEP LEARNING MODEL - "
        f"{network_name}"
    )
    print("=" * 80)

    print(
        f"\nNetwork name : {network_name}"
    )

    print(
        f"Feature file : {FEATURES_FILE}"
    )

    print(
        f"Output folder: {OUTPUT_DIR}"
    )


    # ============================================================
    # 5. LOAD FEATURE DATASET
    # ============================================================

    print(
        "\nLoading feature dataset ..."
    )

    data = pd.read_csv(
        FEATURES_FILE
    )

    print(
        f"Loaded dataset: "
        f"{data.shape[0]} proteins x "
        f"{data.shape[1]} columns"
    )


    # ============================================================
    # 6. SEPARATE FEATURES AND LABEL
    # ============================================================

    feature_cols = [
        c
        for c in data.columns
        if c not in (
            'protein',
            'label'
        )
    ]


    X = data[
        feature_cols
    ].values


    y = data[
        'label'
    ].values


    print(
        f"Number of features: "
        f"{len(feature_cols)}"
    )

    print(
        f"Feature matrix shape: "
        f"{X.shape}"
    )

    print(
        f"Label vector shape: "
        f"{y.shape}"
    )


    # ============================================================
    # 7. TRAIN / VALIDATION / TEST SPLIT
    # ============================================================
    #
    # 70% train
    # 15% validation
    # 15% held-out test
    #
    # Stratified on the label.
    #
    # The same split is saved to SPLITS_FILE.
    #
    # Step 3, Step 4, Step 5 and Step 6
    # will load this exact split.
    # ============================================================

    print(
        "\n" + "-" * 70
    )

    print(
        "Creating train / validation / test split ..."
    )


    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        stratify=y,
        random_state=RANDOM_STATE
    )


    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=RANDOM_STATE
    )


    print(
        f"Train: {X_train.shape[0]}  "
        f"Val: {X_val.shape[0]}  "
        f"Test: {X_test.shape[0]}"
    )


    # ============================================================
    # 8. STANDARDIZE FEATURES
    # ============================================================
    #
    # Fit scaler ONLY on training data.
    #
    # Validation and test data are transformed using
    # the training-fitted scaler.
    # ============================================================

    print(
        "\nStandardizing features ..."
    )


    scaler = StandardScaler().fit(
        X_train
    )


    X_train_s = scaler.transform(
        X_train
    )


    X_val_s = scaler.transform(
        X_val
    )


    X_test_s = scaler.transform(
        X_test
    )


    # ============================================================
    # 9. HYPERPARAMETER TUNING
    # ============================================================

    print(
        "\n" + "-" * 70
    )

    print(
        "Hyperparameter tuning: Basic MLP"
    )

    print(
        "-" * 70
    )


    param_grid = {

        'hidden_layer_sizes': [
            (16,),
            (32,),
            (64,)
        ],

        'alpha': [
            1e-4,
            1e-3,
            1e-2
        ],

        'learning_rate_init': [
            0.001,
            0.01
        ]
    }


    # ============================================================
    # 10. DEFINE BASE MLP
    # ============================================================

    base_mlp = MLPClassifier(

        activation='relu',

        solver='adam',

        max_iter=500,

        early_stopping=True,

        n_iter_no_change=15,

        random_state=RANDOM_STATE
    )


    # ============================================================
    # 11. STRATIFIED CROSS-VALIDATION
    # ============================================================

    cv = StratifiedKFold(

        n_splits=5,

        shuffle=True,

        random_state=RANDOM_STATE
    )


    # ============================================================
    # 12. GRID SEARCH
    # ============================================================

    grid = GridSearchCV(

        estimator=base_mlp,

        param_grid=param_grid,

        scoring='roc_auc',

        cv=cv,

        n_jobs=-1
    )


    grid.fit(

        X_train_s,

        y_train
    )


    # ============================================================
    # 13. DISPLAY GRID SEARCH RESULTS
    # ============================================================

    print(
        "\n=== BASIC MODEL: Grid Search Results ==="
    )


    print(
        "Best params:",
        grid.best_params_
    )


    print(
        f"Best CV ROC-AUC: "
        f"{grid.best_score_:.4f}"
    )


    # ============================================================
    # 14. GET BEST BASIC MODEL
    # ============================================================

    best_basic = grid.best_estimator_


    # ============================================================
    # 15. REFIT BEST MODEL
    # ============================================================

    best_basic.fit(

        X_train_s,

        y_train
    )


    # ============================================================
    # 16. EVALUATION FUNCTION
    # ============================================================

    def evaluate(
        model,
        X_eval,
        y_eval,
        name
    ):

        proba = model.predict_proba(

            X_eval

        )[:, 1]


        pred = model.predict(

            X_eval

        )


        metrics = {

            'accuracy':
                accuracy_score(
                    y_eval,
                    pred
                ),

            'precision':
                precision_score(
                    y_eval,
                    pred,
                    zero_division=0
                ),

            'recall':
                recall_score(
                    y_eval,
                    pred,
                    zero_division=0
                ),

            'f1':
                f1_score(
                    y_eval,
                    pred,
                    zero_division=0
                ),

            'roc_auc':
                roc_auc_score(
                    y_eval,
                    proba
                )
        }


        print(
            f"\n--- {name} ---"
        )


        for k, v in metrics.items():

            print(
                f"{k:>10s}: "
                f"{v:.4f}"
            )


        print(
            "\nConfusion Matrix:"
        )


        print(
            confusion_matrix(
                y_eval,
                pred
            )
        )


        return metrics


    # ============================================================
    # 17. BASIC MODEL PERFORMANCE
    # ============================================================

    print(
        "\n=== BASIC MODEL PERFORMANCE ==="
    )


    val_metrics_basic = evaluate(

        best_basic,

        X_val_s,

        y_val,

        "Validation set"
    )


    test_metrics_basic = evaluate(

        best_basic,

        X_test_s,

        y_test,

        "Held-out test set"
    )


    # ============================================================
    # 18. FULL CLASSIFICATION REPORT
    # ============================================================

    print(
        "\nFull classification report (test set):"
    )


    print(
        classification_report(

            y_test,

            best_basic.predict(
                X_test_s
            ),

            target_names=[
                'non-essential',
                'essential'
            ],

            zero_division=0
        )
    )


    # ============================================================
    # 19. SAVE SCALER
    # ============================================================

    joblib.dump(

        scaler,

        SCALER_FILE
    )


    print(
        f"\nSaved scaler -> "
        f"{SCALER_FILE}"
    )


    # ============================================================
    # 20. SAVE BASIC MODEL
    # ============================================================

    joblib.dump(

        best_basic,

        BASIC_MODEL_FILE
    )


    print(
        f"Saved basic model -> "
        f"{BASIC_MODEL_FILE}"
    )


    # ============================================================
    # 21. SAVE TRAIN / VALIDATION / TEST SPLITS
    # ============================================================

    np.savez(

        SPLITS_FILE,

        X_train=X_train,

        y_train=y_train,

        X_val=X_val,

        y_val=y_val,

        X_test=X_test,

        y_test=y_test
    )


    print(
        f"Saved data splits -> "
        f"{SPLITS_FILE}"
    )


    # ============================================================
    # 22. STEP 2 COMPLETE
    # ============================================================

    print(
        "\n" + "=" * 80
    )

    print(
        f"STEP 2 COMPLETED SUCCESSFULLY FOR "
        f"{network_name}"
    )

    print(
        "=" * 80
    )


    # ============================================================
    # 23. RETURN RESULTS
    # ============================================================

    return {

        'network_name':
            network_name,

        'data':
            data,

        'feature_cols':
            feature_cols,

        'X_train':
            X_train,

        'y_train':
            y_train,

        'X_val':
            X_val,

        'y_val':
            y_val,

        'X_test':
            X_test,

        'y_test':
            y_test,

        'X_train_s':
            X_train_s,

        'X_val_s':
            X_val_s,

        'X_test_s':
            X_test_s,

        'scaler':
            scaler,

        'basic_model':
            best_basic,

        'best_params':
            grid.best_params_,

        'best_cv_auc':
            grid.best_score_,

        'val_metrics':
            val_metrics_basic,

        'test_metrics':
            test_metrics_basic,

        'scaler_file':
            SCALER_FILE,

        'basic_model_file':
            BASIC_MODEL_FILE,

        'splits_file':
            SPLITS_FILE,

        'output_dir':
            OUTPUT_DIR
    }
    print("CELL 3 LOADED SUCCESSFULLY")

In [6]:
# ================================================================
# CELL 3B: EXECUTE STEP 2 FOR ALL FOUR NETWORKS
# ================================================================

STEP2_RESULTS = {}


print("\n")
print("#" * 100)
print("STARTING STEP 2 FOR ALL FOUR NETWORKS")
print("#" * 100)


for network_name in NETWORK_CONFIGS.keys():

    print("\n\n")
    print("#" * 100)
    print(
        f"STARTING STEP 2 FOR NETWORK: {network_name}"
    )
    print("#" * 100)


    # ------------------------------------------------------------
    # Get network-specific configuration
    # ------------------------------------------------------------

    config = NETWORK_CONFIGS[
        network_name
    ]


    # ------------------------------------------------------------
    # Get network-specific paths
    #
    # NETWORK_PATHS structure:
    #
    # NETWORK_PATHS['YDIP'] = {
    #     'output_dir': ...,
    #     'paths': {...}
    # }
    # ------------------------------------------------------------

    output_dir = NETWORK_PATHS[
        network_name
    ][
        'output_dir'
    ]


    paths = NETWORK_PATHS[
        network_name
    ][
        'paths'
    ]


    # ------------------------------------------------------------
    # Run Step 2
    # ------------------------------------------------------------

    STEP2_RESULTS[
        network_name
    ] = run_step2_basic_model(

        network_name=network_name,

        config=config,

        paths={
            **paths,

            # The function expects these keys directly
            'output_dir':
                output_dir
        },

        step1_result=STEP1_RESULTS.get(
            network_name,
            None
        )
    )


# ================================================================
# FINAL STEP 2 SUMMARY
# ================================================================

print("\n\n")
print("=" * 100)
print("STEP 2 COMPLETED FOR ALL NETWORKS")
print("=" * 100)


for network_name, result in STEP2_RESULTS.items():

    print("\n" + "-" * 80)

    print(
        f"Network: {network_name}"
    )

    print(
        f"  Best parameters : "
        f"{result['best_params']}"
    )

    print(
        f"  Best CV ROC-AUC  : "
        f"{result['best_cv_auc']:.4f}"
    )

    print(
        f"  Scaler file      : "
        f"{result['scaler_file']}"
    )

    print(
        f"  Basic model file : "
        f"{result['basic_model_file']}"
    )

    print(
        f"  Splits file      : "
        f"{result['splits_file']}"
    )


print("\n")
print("=" * 100)
print("CELL 3 EXECUTION COMPLETE")
print("=" * 100)



####################################################################################################
STARTING STEP 2 FOR ALL FOUR NETWORKS
####################################################################################################



####################################################################################################
STARTING STEP 2 FOR NETWORK: YDIP
####################################################################################################


STEP 2: BASIC DEEP LEARNING MODEL - YDIP

Network name : YDIP
Feature file : /content/YDIP_results/protein_features.csv
Output folder: /content/YDIP_results

Loading feature dataset ...
Loaded dataset: 4758 proteins x 15 columns
Number of features: 13
Feature matrix shape: (4758, 13)
Label vector shape: (4758,)

----------------------------------------------------------------------
Creating train / validation / test split ...
Train: 3330  Val: 714  Test: 714

Standardizing features ...

-------------------------

In [7]:
# ================================================================
# CELL 4: STEP 3 - ADVANCED DEEP LEARNING MODEL
# ================================================================
#
# Builds and tunes the Advanced Deep Neural Network for the
# CURRENT PPI network.
#
# Automatically used for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# Uses:
#
#     - Train / validation / test splits generated by STEP 2
#     - Network-specific scaler generated by STEP 2
#
# Model:
#
#     - Multiple hidden layers
#     - He initialization
#     - ReLU hidden activations
#     - Sigmoid output
#     - Batch normalization
#     - Dropout regularization
#     - L2 weight decay
#     - Adam optimizer
#     - Mini-batch training
#     - Class-weighted binary cross-entropy
#
# Hyperparameters:
#
#     - Randomized search on validation set
#
# Outputs:
#
#     {NETWORK}_results/advanced_model.pkl
#     {NETWORK}_results/hyperparam_search_results.csv
#
# ================================================================


def run_step3_advanced_deep_learning(
    network_name,
    config,
    paths
):

    # ============================================================
    # 1. GET NETWORK-SPECIFIC PATHS
    # ============================================================
    #
    # IMPORTANT:
    #
    # CELL 1 defines these keys as:
    #
    #     'splits_file'
    #     'scaler_file'
    #
    # NOT:
    #
    #     'splits'
    #     'scaler'
    #
    # ============================================================

    OUTPUT_DIR = paths['output_dir']

    SPLITS_FILE = paths['splits_file']

    SCALER_FILE = paths['scaler_file']

    ADVANCED_MODEL_FILE = paths[
        'advanced_model_file'
    ]

    HYPERPARAM_RESULTS_FILE = paths[
        'advanced_search_file'
    ]


    # ============================================================
    # 2. RANDOM STATE
    # ============================================================

    RANDOM_STATE = 42

    np.random.seed(
        RANDOM_STATE
    )

    random.seed(
        RANDOM_STATE
    )


    # ============================================================
    # 3. CREATE OUTPUT DIRECTORY
    # ============================================================

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ============================================================
    # 4. PRINT NETWORK INFORMATION
    # ============================================================

    print("\n")
    print("=" * 80)

    print(
        f"STEP 3: ADVANCED DEEP LEARNING MODEL - "
        f"{network_name}"
    )

    print("=" * 80)

    print(
        f"\nNetwork name  : {network_name}"
    )

    print(
        f"Output folder : {OUTPUT_DIR}"
    )

    print(
        f"Splits file   : {SPLITS_FILE}"
    )

    print(
        f"Scaler file   : {SCALER_FILE}"
    )

    print(
        f"Model output  : {ADVANCED_MODEL_FILE}"
    )

    print(
        f"Search output : {HYPERPARAM_RESULTS_FILE}"
    )


    # ============================================================
    # 5. VERIFY REQUIRED STEP 2 OUTPUTS
    # ============================================================

    print(
        "\nChecking required STEP 2 outputs..."
    )


    if not os.path.exists(
        SPLITS_FILE
    ):

        raise FileNotFoundError(

            f"\nSTEP 2 split file not found for "
            f"{network_name}.\n\n"

            f"Expected file:\n"
            f"{SPLITS_FILE}\n\n"

            f"Please successfully run STEP 2 for "
            f"{network_name} before running STEP 3."

        )


    print(
        f"[OK] Splits file found:"
    )

    print(
        f"     {SPLITS_FILE}"
    )


    if not os.path.exists(
        SCALER_FILE
    ):

        raise FileNotFoundError(

            f"\nSTEP 2 scaler file not found for "
            f"{network_name}.\n\n"

            f"Expected file:\n"
            f"{SCALER_FILE}\n\n"

            f"Please successfully run STEP 2 for "
            f"{network_name} before running STEP 3."

        )


    print(
        f"[OK] Scaler file found:"
    )

    print(
        f"     {SCALER_FILE}"
    )


    # ============================================================
    # 6. LOAD DATA SPLITS
    # ============================================================

    print(
        "\nLoading train/validation/test splits..."
    )


    splits = np.load(
        SPLITS_FILE
    )


    # ------------------------------------------------------------
    # Verify required arrays
    # ------------------------------------------------------------

    required_split_keys = [

        'X_train',
        'y_train',

        'X_val',
        'y_val',

        'X_test',
        'y_test'

    ]


    for key in required_split_keys:

        if key not in splits:

            raise KeyError(

                f"Required array '{key}' "
                f"was not found in:\n"
                f"{SPLITS_FILE}"

            )


    # ------------------------------------------------------------
    # Load splits
    # ------------------------------------------------------------

    X_train = splits[
        'X_train'
    ]

    y_train = splits[
        'y_train'
    ]


    X_val = splits[
        'X_val'
    ]

    y_val = splits[
        'y_val'
    ]


    X_test = splits[
        'X_test'
    ]

    y_test = splits[
        'y_test'
    ]


    print(
        f"Train: "
        f"{X_train.shape[0]} | "
        f"Validation: "
        f"{X_val.shape[0]} | "
        f"Test: "
        f"{X_test.shape[0]}"
    )


    # ============================================================
    # 7. VERIFY BINARY CLASSIFICATION
    # ============================================================

    unique_labels = np.unique(
        y_train
    )


    print(
        f"\nTraining labels: "
        f"{unique_labels}"
    )


    if len(
        unique_labels
    ) != 2:

        raise ValueError(

            "STEP 3 requires binary classification. "

            f"Found labels: {unique_labels}"

        )


    # ============================================================
    # 8. LOAD NETWORK-SPECIFIC SCALER
    # ============================================================

    print(
        "\nLoading network-specific scaler..."
    )


    scaler = joblib.load(
        SCALER_FILE
    )


    # ============================================================
    # 9. TRANSFORM DATA
    # ============================================================
    #
    # IMPORTANT:
    #
    # The scaler was fitted ONLY on X_train
    # during STEP 2.
    #
    # We reuse that same scaler here.
    #
    # This prevents data leakage.
    #
    # ============================================================

    X_train_s = scaler.transform(
        X_train
    )


    X_val_s = scaler.transform(
        X_val
    )


    X_test_s = scaler.transform(
        X_test
    )


    n_features = X_train_s.shape[
        1
    ]


    print(
        f"Number of input features: "
        f"{n_features}"
    )


    # ============================================================
    # 10. HYPERPARAMETER SEARCH SPACE
    # ============================================================

    search_space = {

        'hidden': [

            (64, 32),

            (128, 64),

            (64, 32, 16)

        ],

        'dropout': [

            0.1,

            0.2,

            0.3

        ],

        'l2': [

            1e-5,

            1e-4,

            1e-3

        ],

        'lr': [

            1e-3,

            3e-3

        ],

        'class_weight': [

            1.0,

            2.0,

            3.0

        ]

    }


    # ============================================================
    # 11. CREATE RANDOMIZED TRIALS
    # ============================================================

    n_trials = 12


    keys = list(
        search_space.keys()
    )


    combos = list(
        itertools.product(
            *search_space.values()
        )
    )


    # ------------------------------------------------------------
    # Make randomized selection reproducible
    # ------------------------------------------------------------

    rng = random.Random(
        RANDOM_STATE
    )


    rng.shuffle(
        combos
    )


    trials = combos[
        :min(
            n_trials,
            len(combos)
        )
    ]


    print(
        f"\nRunning randomized hyperparameter search: "
        f"{len(trials)} trials"
    )


    # ============================================================
    # 12. RUN HYPERPARAMETER SEARCH
    # ============================================================

    results = []


    for i, combo in enumerate(
        trials,
        start=1
    ):

        cfg = dict(
            zip(
                keys,
                combo
            )
        )


        layer_sizes = [

            n_features,

            *cfg['hidden'],

            1

        ]


        print(
            f"\nTrial {i}/{len(trials)}"
        )


        print(
            f"Architecture: "
            f"{layer_sizes}"
        )


        print(
            f"Dropout: "
            f"{cfg['dropout']} | "
            f"L2: "
            f"{cfg['l2']:.0e} | "
            f"LR: "
            f"{cfg['lr']:.0e} | "
            f"Class weight: "
            f"{cfg['class_weight']}"
        )


        # --------------------------------------------------------
        # CREATE MODEL
        # --------------------------------------------------------

        model = DeepNN(

            layer_sizes,

            dropout=cfg[
                'dropout'
            ],

            l2=cfg[
                'l2'
            ],

            lr=cfg[
                'lr'
            ],

            class_weight=cfg[
                'class_weight'
            ]

        )


        # --------------------------------------------------------
        # TRAIN MODEL
        # --------------------------------------------------------

        model.fit(

            X_train_s,

            y_train,

            X_val_s,

            y_val,

            epochs=120,

            batch_size=64,

            patience=15,

            verbose=False

        )


        # --------------------------------------------------------
        # VALIDATION PREDICTIONS
        # --------------------------------------------------------

        val_proba = np.asarray(

            model.predict_proba(

                X_val_s

            )

        ).reshape(
            -1
        )


        # --------------------------------------------------------
        # VALIDATION ROC-AUC
        # --------------------------------------------------------

        if len(
            np.unique(
                y_val
            )
        ) != 2:

            raise ValueError(

                f"Validation set for {network_name} "
                f"does not contain both classes."

            )


        val_auc = roc_auc_score(

            y_val,

            val_proba

        )


        # --------------------------------------------------------
        # STORE RESULTS
        # --------------------------------------------------------

        results.append(

            {

                **cfg,

                'val_auc':
                    val_auc

            }

        )


        print(

            f"Validation ROC-AUC: "
            f"{val_auc:.4f}"

        )


    # ============================================================
    # 13. VERIFY SEARCH RESULTS
    # ============================================================

    if len(
        results
    ) == 0:

        raise RuntimeError(

            f"No hyperparameter search results "
            f"were generated for {network_name}."

        )


    # ============================================================
    # 14. SORT SEARCH RESULTS
    # ============================================================

    results_df = (

        pd.DataFrame(

            results

        )

        .sort_values(

            'val_auc',

            ascending=False

        )

        .reset_index(

            drop=True

        )

    )


    print(

        "\n=== ADVANCED MODEL: "
        "TOP 5 CONFIGURATIONS ==="

    )


    print(

        results_df

        .head(5)

        .to_string(

            index=False

        )

    )


    # ============================================================
    # 15. SELECT BEST CONFIGURATION
    # ============================================================

    best_cfg = (

        results_df

        .iloc[0]

        .to_dict()

    )


    print(

        "\nBest configuration:"

    )


    print(

        best_cfg

    )


    # ============================================================
    # 16. EXTRACT BEST HYPERPARAMETERS
    # ============================================================

    best_hidden = tuple(

        best_cfg[
            'hidden'
        ]

    )


    best_dropout = float(

        best_cfg[
            'dropout'
        ]

    )


    best_l2 = float(

        best_cfg[
            'l2'
        ]

    )


    best_lr = float(

        best_cfg[
            'lr'
        ]

    )


    best_class_weight = float(

        best_cfg[
            'class_weight'
        ]

    )


    layer_sizes = [

        n_features,

        *best_hidden,

        1

    ]


    # ============================================================
    # 17. RETRAIN BEST MODEL
    # ============================================================

    print(

        "\nRetraining best Advanced Deep NN..."

    )


    print(

        f"Architecture: "
        f"{layer_sizes}"

    )


    print(

        f"Dropout: "
        f"{best_dropout} | "

        f"L2: "
        f"{best_l2:.0e} | "

        f"LR: "
        f"{best_lr:.0e} | "

        f"Class weight: "
        f"{best_class_weight}"

    )


    best_advanced = DeepNN(

        layer_sizes,

        dropout=best_dropout,

        l2=best_l2,

        lr=best_lr,

        class_weight=best_class_weight

    )


    history = best_advanced.fit(

        X_train_s,

        y_train,

        X_val_s,

        y_val,

        epochs=300,

        batch_size=64,

        patience=25,

        verbose=True

    )


    # ============================================================
    # 18. EVALUATION FUNCTION
    # ============================================================

    def evaluate(

        model,

        X,

        y,

        name,

        threshold=0.5

    ):

        proba = np.asarray(

            model.predict_proba(

                X

            )

        ).reshape(

            -1

        )


        pred = (

            proba >= threshold

        ).astype(

            int

        )


        metrics = {

            'accuracy':

                accuracy_score(

                    y,

                    pred

                ),

            'precision':

                precision_score(

                    y,

                    pred,

                    zero_division=0

                ),

            'recall':

                recall_score(

                    y,

                    pred,

                    zero_division=0

                ),

            'f1':

                f1_score(

                    y,

                    pred,

                    zero_division=0

                ),

            'roc_auc':

                roc_auc_score(

                    y,

                    proba

                )

        }


        print(

            f"\n--- {name} "
            f"(threshold={threshold}) ---"

        )


        for key, value in metrics.items():

            print(

                f"{key:>10s}: "
                f"{value:.4f}"

            )


        print(

            "\nConfusion Matrix:"

        )


        print(

            confusion_matrix(

                y,

                pred

            )

        )


        return metrics


    # ============================================================
    # 19. MODEL PERFORMANCE
    # ============================================================

    print(

        "\n=== ADVANCED MODEL PERFORMANCE "
        "(DEFAULT THRESHOLD 0.5) ==="

    )


    val_metrics_adv = evaluate(

        best_advanced,

        X_val_s,

        y_val,

        "Validation set"

    )


    test_metrics_adv = evaluate(

        best_advanced,

        X_test_s,

        y_test,

        "Held-out test set"

    )


    # ============================================================
    # 20. CLASSIFICATION REPORT
    # ============================================================

    print(

        "\nFull classification report "
        "(test set):"

    )


    test_proba_adv = np.asarray(

        best_advanced.predict_proba(

            X_test_s

        )

    ).reshape(

        -1

    )


    test_pred_adv = (

        test_proba_adv >= 0.5

    ).astype(

        int

    )


    print(

        classification_report(

            y_test,

            test_pred_adv,

            target_names=[

                'non-essential',

                'essential'

            ],

            zero_division=0

        )

    )


    # ============================================================
    # 21. SAVE ADVANCED MODEL BUNDLE
    # ============================================================

    advanced_bundle = {

        'model':
            best_advanced,

        'best_cfg':
            best_cfg,

        'layer_sizes':
            layer_sizes,

        'n_features':
            n_features,

        'random_state':
            RANDOM_STATE,

        'network_name':
            network_name

    }


    with open(

        ADVANCED_MODEL_FILE,

        'wb'

    ) as f:

        pickle.dump(

            advanced_bundle,

            f

        )


    print(

        f"\nSaved advanced model -> "
        f"{ADVANCED_MODEL_FILE}"

    )


    # ============================================================
    # 22. SAVE HYPERPARAMETER SEARCH RESULTS
    # ============================================================

    results_df.to_csv(

        HYPERPARAM_RESULTS_FILE,

        index=False

    )


    print(

        f"Saved search results -> "
        f"{HYPERPARAM_RESULTS_FILE}"

    )


    # ============================================================
    # 23. FINAL OUTPUT
    # ============================================================

    print(

        "\n" + "=" * 80

    )


    print(

        f"STEP 3 COMPLETED SUCCESSFULLY "
        f"FOR {network_name}"

    )


    print(

        "=" * 80

    )


    # ============================================================
    # 24. RETURN RESULTS
    # ============================================================

    return {

        'network_name':
            network_name,

        'model':
            best_advanced,

        'best_cfg':
            best_cfg,

        'history':
            history,

        'results_df':
            results_df,

        'advanced_model_file':
            ADVANCED_MODEL_FILE,

        'hyperparam_results_file':
            HYPERPARAM_RESULTS_FILE,

        'val_metrics':
            val_metrics_adv,

        'test_metrics':
            test_metrics_adv,

        'n_features':
            n_features

    }

In [8]:
# ================================================================
# STEP 3: RUN ADVANCED DEEP LEARNING FOR ALL NETWORKS
# ================================================================

print("\n")
print("#" * 100)
print(
    "STARTING STEP 3 FOR ALL FOUR NETWORKS"
)
print("#" * 100)


# ================================================================
# INITIALIZE RESULTS
# ================================================================

STEP3_RESULTS = {}


# ================================================================
# RUN FOR EACH NETWORK
# ================================================================

for NETWORK_NAME in NETWORKS:

    print("\n")
    print("#" * 100)

    print(
        f"STARTING STEP 3 FOR NETWORK: "
        f"{NETWORK_NAME}"
    )

    print("#" * 100)


    # ============================================================
    # GET CONFIGURATION
    # ============================================================

    CONFIG = NETWORK_CONFIGS[
        NETWORK_NAME
    ]


    # ============================================================
    # GET FLAT PATH DICTIONARY
    #
    # IMPORTANT:
    #
    # MUST use:
    #
    #     PATHS[NETWORK_NAME]
    #
    # NOT:
    #
    #     NETWORK_PATHS[NETWORK_NAME]
    #
    # ============================================================

    NETWORK_SPECIFIC_PATHS = PATHS[
        NETWORK_NAME
    ]


    # ============================================================
    # DEBUG CHECK
    # ============================================================

    print(
        "\nPATHS being passed to STEP 3:"
    )

    print(
        NETWORK_SPECIFIC_PATHS
    )


    print(
        "\nAvailable path keys:"
    )

    print(
        list(
            NETWORK_SPECIFIC_PATHS.keys()
        )
    )


    # ============================================================
    # VERIFY REQUIRED KEYS
    # ============================================================

    required_keys = [

        'output_dir',

        'splits_file',

        'scaler_file',

        'advanced_model_file',

        'advanced_search_file'

    ]


    missing_keys = [

        key

        for key in required_keys

        if key not in NETWORK_SPECIFIC_PATHS

    ]


    if missing_keys:

        raise KeyError(

            f"\nSTEP 3 PATH CONFIGURATION ERROR\n\n"

            f"Network: {NETWORK_NAME}\n"

            f"Missing keys: {missing_keys}\n\n"

            f"Available keys:\n"
            f"{list(NETWORK_SPECIFIC_PATHS.keys())}\n\n"

            f"Expected flat path dictionary:\n"

            f"PATHS['{NETWORK_NAME}']"

        )


    # ============================================================
    # VERIFY STEP 2 OUTPUTS
    # ============================================================

    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'splits_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nSTEP 2 output missing for "
            f"{NETWORK_NAME}.\n\n"

            f"Expected:\n"

            f"{NETWORK_SPECIFIC_PATHS['splits_file']}\n\n"

            f"Run STEP 2 first."

        )


    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'scaler_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nSTEP 2 scaler missing for "
            f"{NETWORK_NAME}.\n\n"

            f"Expected:\n"

            f"{NETWORK_SPECIFIC_PATHS['scaler_file']}\n\n"

            f"Run STEP 2 first."

        )


    print(
        "\n[OK] STEP 2 outputs verified."
    )


    # ============================================================
    # RUN STEP 3
    # ============================================================

    STEP3_RESULTS[
        NETWORK_NAME
    ] = run_step3_advanced_deep_learning(

        network_name=NETWORK_NAME,

        config=CONFIG,

        paths=NETWORK_SPECIFIC_PATHS

    )


    # ============================================================
    # NETWORK COMPLETE
    # ============================================================

    print("\n")
    print("-" * 100)

    print(
        f"STEP 3 COMPLETED FOR NETWORK: "
        f"{NETWORK_NAME}"
    )

    print("-" * 100)


# ================================================================
# ALL NETWORKS COMPLETE
# ================================================================

print("\n")
print("#" * 100)

print(
    "STEP 3 COMPLETED SUCCESSFULLY "
    "FOR ALL FOUR NETWORKS"
)

print("#" * 100)


print(
    "\nCompleted networks:"
)


for NETWORK_NAME in STEP3_RESULTS:

    print(
        f"  [OK] {NETWORK_NAME}"
    )



####################################################################################################
STARTING STEP 3 FOR ALL FOUR NETWORKS
####################################################################################################


####################################################################################################
STARTING STEP 3 FOR NETWORK: YDIP
####################################################################################################

PATHS being passed to STEP 3:
{'output_dir': '/content/YDIP_results', 'features_file': '/content/YDIP_results/protein_features.csv', 'splits_file': '/content/YDIP_results/splits.npz', 'scaler_file': '/content/YDIP_results/scaler.joblib', 'basic_model_file': '/content/YDIP_results/basic_model.joblib', 'basic_search_file': '/content/YDIP_results/basic_search_results.csv', 'advanced_model_file': '/content/YDIP_results/advanced_model.pkl', 'advanced_search_file': '/content/YDIP_results/hyperparam_search_results.csv', 

In [9]:
# ================================================================
# CELL 5: STEP 4 - VALIDATION & MODEL COMPARISON
# ================================================================
#
# Loads and compares:
#
#     1. Basic MLP
#     2. Advanced Deep NN
#
# Selects the optimal decision threshold using Youden's J statistic
# on the validation set, then evaluates both models on the untouched
# held-out test set.
#
# This function is automatically called for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# It uses the network-specific paths from CELL 1.
#
# Outputs:
#
#     /content/YDIP_results/model_comparison.png
#     /content/YDIP_results/final_comparison_table.csv
#
#     /content/YMIPS_results/model_comparison.png
#     /content/YMIPS_results/final_comparison_table.csv
#
#     /content/YMBD_results/model_comparison.png
#     /content/YMBD_results/final_comparison_table.csv
#
#     /content/YHQ_results/model_comparison.png
#     /content/YHQ_results/final_comparison_table.csv
#
# ================================================================


def run_step4_validation(
    network_name,
    config,
    paths
):

    # ============================================================
    # 1. GET NETWORK-SPECIFIC FILE PATHS
    # ============================================================

    OUTPUT_DIR = paths['output_dir']

    SPLITS_FILE = paths['splits_file']

    SCALER_FILE = paths['scaler_file']

    BASIC_MODEL_FILE = paths['basic_model_file']

    ADVANCED_MODEL_FILE = paths['advanced_model_file']


    # ============================================================
    # 2. CREATE OUTPUT DIRECTORY
    # ============================================================

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ============================================================
    # 3. RANDOM STATE
    # ============================================================

    RANDOM_STATE = 42


    # ============================================================
    # 4. PRINT NETWORK INFORMATION
    # ============================================================

    print("\n")
    print("=" * 80)

    print(
        f"STEP 4: VALIDATION & MODEL COMPARISON - "
        f"{network_name}"
    )

    print("=" * 80)

    print(
        f"\nNetwork name   : {network_name}"
    )

    print(
        f"Output folder  : {OUTPUT_DIR}"
    )

    print(
        f"Splits file    : {SPLITS_FILE}"
    )

    print(
        f"Scaler file    : {SCALER_FILE}"
    )

    print(
        f"Basic model    : {BASIC_MODEL_FILE}"
    )

    print(
        f"Advanced model : {ADVANCED_MODEL_FILE}"
    )


    # ============================================================
    # 5. LOAD DATA SPLITS
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "1. Loading data splits ..."
    )

    print("-" * 70)


    splits = np.load(
        SPLITS_FILE
    )


    X_train = splits[
        'X_train'
    ]

    y_train = splits[
        'y_train'
    ]


    X_val = splits[
        'X_val'
    ]

    y_val = splits[
        'y_val'
    ]


    X_test = splits[
        'X_test'
    ]

    y_test = splits[
        'y_test'
    ]


    print(
        f"Train: {X_train.shape[0]} | "
        f"Validation: {X_val.shape[0]} | "
        f"Test: {X_test.shape[0]}"
    )


    # ============================================================
    # 6. LOAD NETWORK-SPECIFIC SCALER
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "2. Loading network-specific scaler ..."
    )

    print("-" * 70)


    scaler = joblib.load(
        SCALER_FILE
    )


    X_val_s = scaler.transform(
        X_val
    )


    X_test_s = scaler.transform(
        X_test
    )


    print(
        "StandardScaler loaded successfully."
    )


    # ============================================================
    # 7. LOAD BASIC MLP
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "3. Loading Basic MLP model ..."
    )

    print("-" * 70)


    basic_model = joblib.load(
        BASIC_MODEL_FILE
    )


    print(
        "Basic MLP model loaded successfully."
    )


    # ============================================================
    # 8. LOAD ADVANCED DEEP NN
    # ============================================================
    #
    # Step 3 saves the Advanced Deep NN as a bundle:
    #
    # {
    #     'model': best_advanced,
    #     'best_cfg': best_cfg,
    #     'layer_sizes': layer_sizes,
    #     'n_features': n_features,
    #     'random_state': RANDOM_STATE
    # }
    #
    # Therefore the actual model is loaded using ['model'].
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "4. Loading Advanced Deep NN model ..."
    )

    print("-" * 70)


    with open(
        ADVANCED_MODEL_FILE,
        'rb'
    ) as f:

        advanced_bundle = pickle.load(
            f
        )


    advanced_model = (
        advanced_bundle['model']
    )


    print(
        "Advanced Deep NN model loaded successfully."
    )


    # ============================================================
    # 9. HELPER FUNCTION:
    #    CONVERT PREDICTIONS TO 1D PROBABILITIES
    # ============================================================

    def get_positive_probability(
        model,
        X
    ):
        """
        Returns the probability of the positive class
        (essential).

        Handles:
            (n_samples,)
            (n_samples, 1)
            (n_samples, 2)
        """

        proba = np.asarray(
            model.predict_proba(
                X
            )
        )


        if proba.ndim == 1:

            return proba.reshape(
                -1
            )


        if proba.ndim == 2:

            if proba.shape[1] == 1:

                return proba[:, 0]


            if proba.shape[1] == 2:

                return proba[:, 1]


        raise ValueError(
            f"Unexpected predict_proba output shape: "
            f"{proba.shape}"
        )


    # ============================================================
    # 10. FIND BEST DECISION THRESHOLD
    # ============================================================
    #
    # Youden's J statistic:
    #
    #     J = TPR - FPR
    #
    # The threshold is determined ONLY from validation predictions.
    #
    # The test set is never used to determine the threshold.
    # ============================================================

    def best_threshold(
        y_true,
        proba
    ):

        fpr, tpr, thresholds = roc_curve(
            y_true,
            proba
        )


        j_scores = (
            tpr - fpr
        )


        best_index = np.argmax(
            j_scores
        )


        return float(
            thresholds[
                best_index
            ]
        )


    # ============================================================
    # 11. FULL EVALUATION FUNCTION
    # ============================================================

    def full_eval(
        y_true,
        proba,
        threshold
    ):

        pred = (
            proba >= threshold
        ).astype(
            int
        )


        metrics = {

            'threshold':
                float(
                    threshold
                ),

            'accuracy':
                accuracy_score(
                    y_true,
                    pred
                ),

            'precision':
                precision_score(
                    y_true,
                    pred,
                    zero_division=0
                ),

            'recall':
                recall_score(
                    y_true,
                    pred,
                    zero_division=0
                ),

            'f1':
                f1_score(
                    y_true,
                    pred,
                    zero_division=0
                ),

            'mcc':
                matthews_corrcoef(
                    y_true,
                    pred
                ),

            'roc_auc':
                roc_auc_score(
                    y_true,
                    proba
                )
        }


        cm = confusion_matrix(
            y_true,
            pred
        )


        return (
            metrics,
            cm
        )


    # ============================================================
    # 12. MODEL REGISTRY
    # ============================================================

    models = [

        (
            'Basic MLP',
            basic_model
        ),

        (
            'Advanced Deep NN',
            advanced_model
        )

    ]


    # ============================================================
    # 13. INITIALIZE RESULTS
    # ============================================================

    results = {}


    # ============================================================
    # 14. CREATE ROC / PR FIGURE
    # ============================================================

    plt.figure(
        figsize=(12, 5)
    )


    # ============================================================
    # 15. EVALUATE ALL MODELS
    # ============================================================

    for name, model in models:

        print(
            f"\n{'=' * 70}"
        )

        print(
            name
        )

        print(
            f"{'=' * 70}"
        )


        # --------------------------------------------------------
        # Validation probabilities
        # --------------------------------------------------------

        val_proba = (
            get_positive_probability(
                model,
                X_val_s
            )
        )


        # --------------------------------------------------------
        # Determine optimal threshold from validation set
        # --------------------------------------------------------

        threshold = best_threshold(
            y_val,
            val_proba
        )


        # --------------------------------------------------------
        # Test probabilities
        # --------------------------------------------------------

        test_proba = (
            get_positive_probability(
                model,
                X_test_s
            )
        )


        # --------------------------------------------------------
        # Default threshold = 0.5
        # --------------------------------------------------------

        metrics_default, cm_default = (
            full_eval(
                y_test,
                test_proba,
                0.5
            )
        )


        # --------------------------------------------------------
        # Validation-tuned threshold
        # --------------------------------------------------------

        metrics_tuned, cm_tuned = (
            full_eval(
                y_test,
                test_proba,
                threshold
            )
        )


        # --------------------------------------------------------
        # Store results
        # --------------------------------------------------------

        results[name] = {

            'default':
                metrics_default,

            'tuned':
                metrics_tuned,

            'cm_default':
                cm_default,

            'cm_tuned':
                cm_tuned,

            'threshold':
                threshold

        }


        # ========================================================
        # ROC CURVE
        # ========================================================

        fpr, tpr, _ = roc_curve(
            y_test,
            test_proba
        )


        plt.subplot(
            1,
            2,
            1
        )


        plt.plot(
            fpr,
            tpr,
            label=(
                f"{name} "
                f"(AUC="
                f"{metrics_default['roc_auc']:.3f})"
            )
        )


        # ========================================================
        # PRECISION-RECALL CURVE
        # ========================================================

        precision, recall, _ = (
            precision_recall_curve(
                y_test,
                test_proba
            )
        )


        plt.subplot(
            1,
            2,
            2
        )


        plt.plot(
            recall,
            precision,
            label=name
        )


        # ========================================================
        # PRINT RESULTS
        # ========================================================

        print(
            f"\nOptimal Youden's J threshold: "
            f"{threshold:.4f}"
        )


        print(
            "\n-- Test set @ default threshold 0.5 --"
        )


        for key, value in (
            metrics_default.items()
        ):

            print(
                f"{key:>10s}: {value:.4f}"
            )


        print(
            "\nConfusion matrix:"
        )


        print(
            cm_default
        )


        print(
            f"\n-- Test set @ tuned threshold "
            f"{threshold:.4f} --"
        )


        for key, value in (
            metrics_tuned.items()
        ):

            print(
                f"{key:>10s}: {value:.4f}"
            )


        print(
            "\nConfusion matrix:"
        )


        print(
            cm_tuned
        )


    # ============================================================
    # 16. FORMAT ROC CURVE
    # ============================================================

    plt.subplot(
        1,
        2,
        1
    )


    plt.plot(
        [0, 1],
        [0, 1],
        'k--',
        alpha=0.3
    )


    plt.xlabel(
        'False Positive Rate'
    )


    plt.ylabel(
        'True Positive Rate'
    )


    plt.title(
        f'ROC Curve — {network_name} Test Set'
    )


    plt.legend()


    # ============================================================
    # 17. FORMAT PRECISION-RECALL CURVE
    # ============================================================

    plt.subplot(
        1,
        2,
        2
    )


    plt.xlabel(
        'Recall'
    )


    plt.ylabel(
        'Precision'
    )


    plt.title(
        f'Precision-Recall Curve — '
        f'{network_name} Test Set'
    )


    plt.legend()


    # ============================================================
    # 18. SAVE MODEL COMPARISON PLOT
    # ============================================================

    plt.tight_layout()


    PLOT_FILE = os.path.join(
        OUTPUT_DIR,
        'model_comparison.png'
    )


    plt.savefig(
        PLOT_FILE,
        dpi=150,
        bbox_inches='tight'
    )


    plt.close()


    print(
        f"\nSaved plot -> {PLOT_FILE}"
    )


    # ============================================================
    # 19. CREATE SUMMARY COMPARISON TABLE
    # ============================================================

    rows = []


    for name, result in (
        results.items()
    ):

        # --------------------------------------------------------
        # Default threshold row
        # --------------------------------------------------------

        default_metrics = (
            result['default']
        )


        rows.append(
            {

                'network':
                    network_name,

                'model':
                    name,

                'threshold_setting':
                    'default_threshold(0.5)',

                **default_metrics

            }
        )


        # --------------------------------------------------------
        # Tuned threshold row
        # --------------------------------------------------------

        tuned_metrics = (
            result['tuned']
        )


        rows.append(
            {

                'network':
                    network_name,

                'model':
                    name,

                'threshold_setting':
                    (
                        f"tuned_threshold("
                        f"{result['threshold']:.3f})"
                    ),

                **tuned_metrics

            }
        )


    summary = pd.DataFrame(
        rows
    )


    # ============================================================
    # 20. SAVE FINAL COMPARISON TABLE
    # ============================================================

    COMPARISON_FILE = os.path.join(
        OUTPUT_DIR,
        'final_comparison_table.csv'
    )


    summary.to_csv(
        COMPARISON_FILE,
        index=False
    )


    # ============================================================
    # 21. PRINT FINAL COMPARISON TABLE
    # ============================================================

    print(
        "\n=== FINAL COMPARISON TABLE ==="
    )


    print(
        summary
        .round(4)
        .to_string(
            index=False
        )
    )


    # ============================================================
    # 22. FINAL OUTPUT
    # ============================================================

    print(
        "\n" + "=" * 80
    )


    print(
        f"STEP 4 COMPLETED SUCCESSFULLY FOR "
        f"{network_name}"
    )


    print(
        "=" * 80
    )


    print(
        "\nSaved files:"
    )


    print(
        f"Model comparison plot -> {PLOT_FILE}"
    )


    print(
        f"Comparison table      -> {COMPARISON_FILE}"
    )


    print(
        "\nCurrent network:"
    )


    print(
        f"Network name -> {network_name}"
    )


    print(
        f"OUTPUT_DIR -> {OUTPUT_DIR}"
    )


    # ============================================================
    # 23. RETURN RESULTS
    # ============================================================

    return {

        'network_name':
            network_name,

        'results':
            results,

        'summary':
            summary,

        'plot_file':
            PLOT_FILE,

        'comparison_file':
            COMPARISON_FILE,

        'basic_model':
            basic_model,

        'advanced_model':
            advanced_model

    }

In [10]:
# ================================================================
# STEP 4: RUN VALIDATION & MODEL COMPARISON
#         FOR ALL FOUR NETWORKS
# ================================================================

print("\n")
print("#" * 100)

print(
    "STARTING STEP 4 FOR ALL FOUR NETWORKS"
)

print("#" * 100)


# ================================================================
# INITIALIZE STEP 4 RESULTS
# ================================================================

STEP4_RESULTS = {}


# ================================================================
# RUN STEP 4 FOR EACH NETWORK
# ================================================================

for NETWORK_NAME in NETWORKS:

    print("\n")
    print("#" * 100)

    print(
        f"STARTING STEP 4 FOR NETWORK: "
        f"{NETWORK_NAME}"
    )

    print("#" * 100)


    # ============================================================
    # GET NETWORK-SPECIFIC CONFIGURATION
    # ============================================================

    CONFIG = NETWORK_CONFIGS[
        NETWORK_NAME
    ]


    # ============================================================
    # GET FLAT NETWORK-SPECIFIC PATHS
    #
    # IMPORTANT:
    #
    # PATHS[NETWORK_NAME] contains:
    #
    #     output_dir
    #     features_file
    #     splits_file
    #     scaler_file
    #     basic_model_file
    #     advanced_model_file
    #     ...
    #
    # DO NOT use:
    #
    #     NETWORK_PATHS[NETWORK_NAME]
    #
    # because that contains an additional 'paths' wrapper.
    # ============================================================

    NETWORK_SPECIFIC_PATHS = PATHS[
        NETWORK_NAME
    ]


    # ============================================================
    # VERIFY REQUIRED STEP 4 PATHS
    # ============================================================

    required_keys = [

        'output_dir',

        'splits_file',

        'scaler_file',

        'basic_model_file',

        'advanced_model_file'

    ]


    missing_keys = [

        key

        for key in required_keys

        if key not in NETWORK_SPECIFIC_PATHS

    ]


    if missing_keys:

        raise KeyError(

            f"\nSTEP 4 PATH CONFIGURATION ERROR\n\n"

            f"Network: {NETWORK_NAME}\n"

            f"Missing keys: {missing_keys}\n\n"

            f"Available keys:\n"
            f"{list(NETWORK_SPECIFIC_PATHS.keys())}"

        )


    # ============================================================
    # VERIFY STEP 2 SPLITS FILE
    # ============================================================

    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'splits_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nSTEP 2 split file not found "
            f"for {NETWORK_NAME}.\n\n"

            f"Expected file:\n"

            f"{NETWORK_SPECIFIC_PATHS['splits_file']}\n\n"

            f"Please run STEP 2 before STEP 4."

        )


    # ============================================================
    # VERIFY STEP 2 SCALER
    # ============================================================

    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'scaler_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nSTEP 2 scaler not found "
            f"for {NETWORK_NAME}.\n\n"

            f"Expected file:\n"

            f"{NETWORK_SPECIFIC_PATHS['scaler_file']}\n\n"

            f"Please run STEP 2 before STEP 4."

        )


    # ============================================================
    # VERIFY BASIC MODEL
    # ============================================================

    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'basic_model_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nBasic MLP model not found "
            f"for {NETWORK_NAME}.\n\n"

            f"Expected file:\n"

            f"{NETWORK_SPECIFIC_PATHS['basic_model_file']}\n\n"

            f"Please run STEP 2 before STEP 4."

        )


    # ============================================================
    # VERIFY ADVANCED MODEL
    # ============================================================

    if not os.path.exists(

        NETWORK_SPECIFIC_PATHS[
            'advanced_model_file'
        ]

    ):

        raise FileNotFoundError(

            f"\nAdvanced Deep NN model not found "
            f"for {NETWORK_NAME}.\n\n"

            f"Expected file:\n"

            f"{NETWORK_SPECIFIC_PATHS['advanced_model_file']}\n\n"

            f"Please run STEP 3 before STEP 4."

        )


    # ============================================================
    # PRINT VERIFIED FILES
    # ============================================================

    print(
        "\n[OK] Required files verified."
    )


    print(
        f"  Splits file    : "
        f"{NETWORK_SPECIFIC_PATHS['splits_file']}"
    )


    print(
        f"  Scaler file    : "
        f"{NETWORK_SPECIFIC_PATHS['scaler_file']}"
    )


    print(
        f"  Basic model    : "
        f"{NETWORK_SPECIFIC_PATHS['basic_model_file']}"
    )


    print(
        f"  Advanced model : "
        f"{NETWORK_SPECIFIC_PATHS['advanced_model_file']}"
    )


    # ============================================================
    # CALL STEP 4
    # ============================================================

    STEP4_RESULTS[
        NETWORK_NAME
    ] = run_step4_validation(

        network_name=NETWORK_NAME,

        config=CONFIG,

        paths=NETWORK_SPECIFIC_PATHS

    )


    # ============================================================
    # CURRENT NETWORK COMPLETE
    # ============================================================

    print("\n")
    print("-" * 100)

    print(
        f"STEP 4 COMPLETED FOR NETWORK: "
        f"{NETWORK_NAME}"
    )

    print("-" * 100)


# ================================================================
# ALL NETWORKS COMPLETE
# ================================================================

print("\n")
print("#" * 100)

print(
    "STEP 4 COMPLETED SUCCESSFULLY "
    "FOR ALL FOUR NETWORKS"
)

print("#" * 100)


# ================================================================
# DISPLAY COMPLETED NETWORKS
# ================================================================

print(
    "\nCompleted networks:"
)


for NETWORK_NAME in STEP4_RESULTS:

    print(
        f"  [OK] {NETWORK_NAME}"
    )


# ================================================================
# DISPLAY OUTPUT FILES
# ================================================================

print(
    "\nGenerated Step 4 outputs:"
)


for NETWORK_NAME, RESULT in STEP4_RESULTS.items():

    print(
        f"\n{NETWORK_NAME}:"
    )

    print(
        f"  Plot  : "
        f"{RESULT['plot_file']}"
    )

    print(
        f"  Table : "
        f"{RESULT['comparison_file']}"
    )



####################################################################################################
STARTING STEP 4 FOR ALL FOUR NETWORKS
####################################################################################################


####################################################################################################
STARTING STEP 4 FOR NETWORK: YDIP
####################################################################################################

[OK] Required files verified.
  Splits file    : /content/YDIP_results/splits.npz
  Scaler file    : /content/YDIP_results/scaler.joblib
  Basic model    : /content/YDIP_results/basic_model.joblib
  Advanced model : /content/YDIP_results/advanced_model.pkl


STEP 4: VALIDATION & MODEL COMPARISON - YDIP

Network name   : YDIP
Output folder  : /content/YDIP_results
Splits file    : /content/YDIP_results/splits.npz
Scaler file    : /content/YDIP_results/scaler.joblib
Basic model    : /content/YDIP_results/basic_model

In [11]:
# ================================================================
# CELL 6: STEP 5 - ADAM-OPTIMIZED DNN + QUANTUM VQC
# ================================================================
#
# Adds two additional models:
#
#   (A) Adam-Optimized DNN
#   (B) Quantum Variational Quantum Classifier (VQC)
#
# Automatically runs for:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# Network-specific paths are obtained from:
#
#     config
#     paths
#
# from CELL 1.
#
# No manual changing of:
#
#     OUTPUT_DIR
#     SPLITS_FILE
#     SCALER_FILE
#
# is required.
#
# Outputs:
#
#     {OUTPUT_DIR}/adam_plain_model.pkl
#     {OUTPUT_DIR}/quantum_vqc_model.pkl
#     {OUTPUT_DIR}/adam_search_results.csv
#     {OUTPUT_DIR}/quantum_search_results.csv
#
# ================================================================


def run_step5_adam_quantum(
    network_name,
    config,
    paths
):

    # ============================================================
    # 1. GET NETWORK-SPECIFIC PATHS
    # ============================================================

    OUTPUT_DIR = paths['output_dir']

    SPLITS_FILE = paths['splits_file']

    SCALER_FILE = paths['scaler_file']


    # ============================================================
    # 2. IMPORTS
    # ============================================================

    import os
    import sys
    import numpy as np
    import pandas as pd
    import itertools
    import random
    import pickle
    import joblib

    from sklearn.decomposition import PCA


    # ============================================================
    # 3. CUSTOM MODULE PATH
    # ============================================================

    WORK_DIR = '/content'

    if WORK_DIR not in sys.path:

        sys.path.insert(
            0,
            WORK_DIR
        )


    # ============================================================
    # 4. IMPORT CUSTOM MODELS
    # ============================================================

    from adam_plain_nn import AdamPlainNN

    from quantum_vqc import VQC


    # ============================================================
    # 5. RANDOM STATE
    # ============================================================

    RANDOM_STATE = 42

    np.random.seed(
        RANDOM_STATE
    )

    random.seed(
        RANDOM_STATE
    )


    # ============================================================
    # 6. CREATE OUTPUT DIRECTORY
    # ============================================================

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )


    # ============================================================
    # 7. DISPLAY CURRENT NETWORK
    # ============================================================

    print("\n")

    print("=" * 80)

    print(
        f"STEP 5: ADAM-OPTIMIZED DNN + QUANTUM VQC - "
        f"{network_name}"
    )

    print("=" * 80)


    print(
        f"\nNetwork name : {network_name}"
    )

    print(
        f"Splits file  : {SPLITS_FILE}"
    )

    print(
        f"Scaler file  : {SCALER_FILE}"
    )

    print(
        f"Output folder: {OUTPUT_DIR}"
    )


    # ============================================================
    # 8. LOAD TRAIN / VALIDATION / TEST SPLITS
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "1. Loading network-specific "
        "train/validation/test splits"
    )

    print("-" * 70)


    splits = np.load(
        SPLITS_FILE
    )


    X_train = splits[
        'X_train'
    ]

    y_train = splits[
        'y_train'
    ]


    X_val = splits[
        'X_val'
    ]

    y_val = splits[
        'y_val'
    ]


    X_test = splits[
        'X_test'
    ]

    y_test = splits[
        'y_test'
    ]


    print(
        f"Train samples      : "
        f"{len(X_train)}"
    )

    print(
        f"Validation samples : "
        f"{len(X_val)}"
    )

    print(
        f"Test samples       : "
        f"{len(X_test)}"
    )


    # ============================================================
    # 9. LOAD SAME SCALER USED BY STEP 2
    # ============================================================

    print("\n" + "-" * 70)

    print(
        "2. Loading network-specific StandardScaler"
    )

    print("-" * 70)


    scaler = joblib.load(
        SCALER_FILE
    )


    X_train_s = scaler.transform(
        X_train
    )


    X_val_s = scaler.transform(
        X_val
    )


    X_test_s = scaler.transform(
        X_test
    )


    n_features = X_train_s.shape[1]


    print(
        f"Number of input features: "
        f"{n_features}"
    )


    # ============================================================
    # ============================================================
    #                 MODEL A: ADAM-OPTIMIZED DNN
    # ============================================================
    # ============================================================


    print("\n" + "=" * 80)

    print(
        f"(A) ADAM-OPTIMIZED DNN - {network_name}"
    )

    print("=" * 80)


    # ============================================================
    # 10. ADAM DNN HYPERPARAMETER SEARCH SPACE
    # ============================================================

    adam_search_space = {

        'hidden': [

            (32,),

            (64, 32),

            (128, 64)

        ],

        'lr': [

            1e-3,

            3e-3,

            1e-2

        ],

        'class_weight': [

            1.0,

            2.0,

            3.0

        ]

    }


    # ============================================================
    # 11. GENERATE RANDOMIZED SEARCH CONFIGURATIONS
    # ============================================================

    adam_keys = list(
        adam_search_space.keys()
    )


    adam_combos = list(

        itertools.product(

            *adam_search_space.values()

        )

    )


    random.shuffle(
        adam_combos
    )


    N_ADAM_TRIALS = 6


    adam_trials = adam_combos[
        :min(
            N_ADAM_TRIALS,
            len(adam_combos)
        )
    ]


    print(
        f"\nAdam DNN hyperparameter trials: "
        f"{len(adam_trials)}"
    )


    # ============================================================
    # 12. RUN ADAM DNN HYPERPARAMETER SEARCH
    # ============================================================

    adam_results = []


    for i, combo in enumerate(
        adam_trials
    ):

        cfg = dict(

            zip(

                adam_keys,

                combo

            )

        )


        print(
            "\n" + "-" * 60
        )

        print(
            f"Adam DNN trial "
            f"{i + 1}/{len(adam_trials)}"
        )


        print(
            f"Hidden layers : "
            f"{cfg['hidden']}"
        )

        print(
            f"Learning rate : "
            f"{cfg['lr']}"
        )

        print(
            f"Class weight  : "
            f"{cfg['class_weight']}"
        )


        # --------------------------------------------------------
        # Build model
        # --------------------------------------------------------

        layer_sizes = [

            n_features,

            *cfg['hidden'],

            1

        ]


        model = AdamPlainNN(

            layer_sizes,

            lr=cfg['lr'],

            seed=RANDOM_STATE

        )


        # --------------------------------------------------------
        # Train model
        # --------------------------------------------------------

        val_auc = model.fit(

            X_train_s,

            y_train,

            X_val_s,

            y_val,

            epochs=120,

            batch_size=64,

            class_weight=cfg['class_weight'],

            patience=15

        )


        # --------------------------------------------------------
        # Store results
        # --------------------------------------------------------

        adam_results.append(

            {

                **cfg,

                'val_auc':
                    float(
                        val_auc
                    )

            }

        )


        print(

            f"\nTrial result: "
            f"Validation ROC-AUC = "
            f"{val_auc:.4f}"

        )


    # ============================================================
    # 13. SORT ADAM SEARCH RESULTS
    # ============================================================

    adam_results_df = (

        pd.DataFrame(
            adam_results
        )

        .sort_values(

            'val_auc',

            ascending=False

        )

        .reset_index(
            drop=True
        )

    )


    print(
        "\n" + "-" * 70
    )

    print(
        "ADAM DNN HYPERPARAMETER SEARCH RESULTS"
    )

    print(
        "-" * 70
    )


    print(

        adam_results_df

        .head(5)

        .to_string(
            index=False
        )

    )


    # ============================================================
    # 14. SELECT BEST ADAM CONFIGURATION
    # ============================================================

    best_adam_cfg = (

        adam_results_df

        .iloc[0]

        .to_dict()

    )


    print(
        "\nBest Adam DNN configuration:"
    )

    print(
        best_adam_cfg
    )


    # ============================================================
    # 15. RETRAIN BEST ADAM DNN
    # ============================================================

    print(
        "\nRetraining best Adam DNN configuration ..."
    )


    best_adam_layer_sizes = [

        n_features,

        *best_adam_cfg['hidden'],

        1

    ]


    best_adam_model = AdamPlainNN(

        best_adam_layer_sizes,

        lr=float(
            best_adam_cfg['lr']
        ),

        seed=RANDOM_STATE

    )


    best_adam_model.fit(

        X_train_s,

        y_train,

        X_val_s,

        y_val,

        epochs=300,

        batch_size=64,

        class_weight=float(
            best_adam_cfg['class_weight']
        ),

        patience=25,

        verbose=True

    )


    print(
        "\nBest Adam DNN training completed."
    )


    # ============================================================
    # ============================================================
    #                       MODEL B: QUANTUM VQC
    # ============================================================
    # ============================================================


    print(
        "\n" + "=" * 80
    )

    print(
        f"(B) QUANTUM VQC - {network_name}"
    )

    print(
        "=" * 80
    )


    # ============================================================
    # 16. NUMBER OF QUBITS
    # ============================================================

    N_QUBITS = 4


    print(
        f"\nNumber of qubits: "
        f"{N_QUBITS}"
    )


    # ============================================================
    # 17. FIT PCA ONLY ON TRAINING DATA
    # ============================================================

    print(
        "\nFitting PCA on training data ..."
    )


    pca = PCA(

        n_components=N_QUBITS,

        random_state=RANDOM_STATE

    )


    pca.fit(
        X_train_s
    )


    explained_variance = (

        pca.explained_variance_ratio_

    )


    print(

        f"PCA explained variance ratio: "
        f"{explained_variance.round(4)}"

    )


    print(

        f"Cumulative explained variance: "
        f"{explained_variance.sum():.4f}"

    )


    # ============================================================
    # 18. FIT PCA ANGLE SCALING ONLY ON TRAINING DATA
    # ============================================================

    X_train_pca = pca.transform(
        X_train_s
    )


    PCA_ANGLE_SCALE = (

        np.max(

            np.abs(
                X_train_pca
            ),

            axis=0

        )

        + 1e-8

    )


    # ============================================================
    # 19. PCA + ANGLE ENCODING FUNCTION
    # ============================================================

    def to_angles(
        X
    ):

        """

        Transform standardized features into PCA space
        and scale PCA components into [-pi, pi].

        PCA and angle scaling are fitted ONLY on training data.

        The same transformation is applied to validation
        and test data.

        """

        Xp = pca.transform(
            X
        )


        Xp = (

            Xp /

            PCA_ANGLE_SCALE

        )


        Xp = np.clip(

            Xp,

            -1.0,

            1.0

        )


        return (

            Xp * np.pi

        )


    # ============================================================
    # 20. TRANSFORM TRAIN / VALIDATION / TEST
    # ============================================================

    Xq_train = to_angles(
        X_train_s
    )


    Xq_val = to_angles(
        X_val_s
    )


    Xq_test = to_angles(
        X_test_s
    )


    print(
        "\nQuantum input shapes:"
    )


    print(
        f"Quantum train      : "
        f"{Xq_train.shape}"
    )


    print(
        f"Quantum validation : "
        f"{Xq_val.shape}"
    )


    print(
        f"Quantum test       : "
        f"{Xq_test.shape}"
    )


    # ============================================================
    # 21. QUANTUM HYPERPARAMETER SEARCH SPACE
    # ============================================================

    quantum_search_space = {

        'depth': [

            1,

            2

        ],

        'lr': [

            0.05,

            0.1

        ],

        'class_weight': [

            1.0,

            2.0

        ]

    }


    quantum_keys = list(

        quantum_search_space.keys()

    )


    quantum_combos = list(

        itertools.product(

            *quantum_search_space.values()

        )

    )


    random.shuffle(
        quantum_combos
    )


    N_QUANTUM_TRIALS = 4


    quantum_trials = quantum_combos[

        :min(

            N_QUANTUM_TRIALS,

            len(quantum_combos)

        )

    ]


    print(
        f"\nQuantum VQC hyperparameter trials: "
        f"{len(quantum_trials)}"
    )


    # ============================================================
    # 22. CREATE QUANTUM SEARCH SUBSAMPLE
    # ============================================================

    QUANTUM_SUBSAMPLE_SIZE = 300


    subsample_size = min(

        QUANTUM_SUBSAMPLE_SIZE,

        len(Xq_train)

    )


    sub_rng = np.random.RandomState(

        RANDOM_STATE

    )


    sub_idx = sub_rng.choice(

        len(Xq_train),

        size=subsample_size,

        replace=False

    )


    Xq_train_sub = Xq_train[
        sub_idx
    ]


    y_train_sub = y_train[
        sub_idx
    ]


    print(

        f"\nQuantum search training samples: "
        f"{len(Xq_train_sub)}"

    )


    # ============================================================
    # 23. RUN QUANTUM HYPERPARAMETER SEARCH
    # ============================================================

    quantum_results = []


    for i, combo in enumerate(

        quantum_trials

    ):

        cfg = dict(

            zip(

                quantum_keys,

                combo

            )

        )


        print(
            "\n" + "-" * 60
        )

        print(
            f"Quantum VQC trial "
            f"{i + 1}/{len(quantum_trials)}"
        )


        print(
            f"Depth        : "
            f"{cfg['depth']}"
        )

        print(
            f"Learning rate: "
            f"{cfg['lr']}"
        )

        print(
            f"Class weight : "
            f"{cfg['class_weight']}"
        )


        # --------------------------------------------------------
        # Build VQC
        # --------------------------------------------------------

        vqc = VQC(

            n_qubits=N_QUBITS,

            depth=int(
                cfg['depth']
            ),

            lr=float(
                cfg['lr']
            ),

            seed=RANDOM_STATE

        )


        # --------------------------------------------------------
        # Train VQC
        # --------------------------------------------------------

        val_auc = vqc.fit(

            Xq_train_sub,

            y_train_sub,

            Xq_val,

            y_val,

            epochs=6,

            batch_size=16,

            class_weight=float(
                cfg['class_weight']
            ),

            patience=4

        )


        # --------------------------------------------------------
        # Store result
        # --------------------------------------------------------

        quantum_results.append(

            {

                **cfg,

                'val_auc':
                    float(
                        val_auc
                    )

            }

        )


        print(

            f"\nTrial result: "
            f"Validation ROC-AUC = "
            f"{val_auc:.4f}"

        )


    # ============================================================
    # 24. SORT QUANTUM SEARCH RESULTS
    # ============================================================

    quantum_results_df = (

        pd.DataFrame(
            quantum_results
        )

        .sort_values(

            'val_auc',

            ascending=False

        )

        .reset_index(
            drop=True
        )

    )


    print(
        "\n" + "-" * 70
    )

    print(
        "QUANTUM VQC HYPERPARAMETER SEARCH RESULTS"
    )

    print(
        "-" * 70
    )


    print(

        quantum_results_df

        .head(5)

        .to_string(
            index=False
        )

    )


    # ============================================================
    # 25. SELECT BEST QUANTUM CONFIGURATION
    # ============================================================

    best_q_cfg = (

        quantum_results_df

        .iloc[0]

        .to_dict()

    )


    print(
        "\nBest Quantum VQC configuration:"
    )

    print(
        best_q_cfg
    )


    # ============================================================
    # 26. RETRAIN BEST QUANTUM VQC
    # ============================================================

    print(
        "\nRetraining best Quantum VQC configuration ..."
    )


    best_vqc = VQC(

        n_qubits=N_QUBITS,

        depth=int(
            best_q_cfg['depth']
        ),

        lr=float(
            best_q_cfg['lr']
        ),

        seed=RANDOM_STATE

    )


    best_vqc.fit(

        Xq_train_sub,

        y_train_sub,

        Xq_val,

        y_val,

        epochs=15,

        batch_size=16,

        class_weight=float(
            best_q_cfg['class_weight']
        ),

        patience=6,

        verbose=True

    )


    print(
        "\nBest Quantum VQC training completed."
    )


    # ============================================================
    # 27. SAVE ADAM DNN MODEL
    # ============================================================

    ADAM_MODEL_FILE = os.path.join(

        OUTPUT_DIR,

        'adam_plain_model.pkl'

    )


    with open(

        ADAM_MODEL_FILE,

        'wb'

    ) as f:

        pickle.dump(

            {

                'model':
                    best_adam_model,

                'best_cfg':
                    best_adam_cfg,

                'network_name':
                    network_name

            },

            f

        )


    # ============================================================
    # 28. SAVE QUANTUM VQC + PCA INFORMATION
    # ============================================================

    QUANTUM_MODEL_FILE = os.path.join(

        OUTPUT_DIR,

        'quantum_vqc_model.pkl'

    )


    with open(

        QUANTUM_MODEL_FILE,

        'wb'

    ) as f:

        pickle.dump(

            {

                'model':
                    best_vqc,

                'best_cfg':
                    best_q_cfg,

                'pca':
                    pca,

                'pca_angle_scale':
                    PCA_ANGLE_SCALE,

                'n_qubits':
                    N_QUBITS,

                'network_name':
                    network_name

            },

            f

        )


    # ============================================================
    # 29. SAVE ADAM SEARCH RESULTS
    # ============================================================

    ADAM_RESULTS_FILE = os.path.join(

        OUTPUT_DIR,

        'adam_search_results.csv'

    )


    adam_results_df.to_csv(

        ADAM_RESULTS_FILE,

        index=False

    )


    # ============================================================
    # 30. SAVE QUANTUM SEARCH RESULTS
    # ============================================================

    QUANTUM_RESULTS_FILE = os.path.join(

        OUTPUT_DIR,

        'quantum_search_results.csv'

    )


    quantum_results_df.to_csv(

        QUANTUM_RESULTS_FILE,

        index=False

    )


    # ============================================================
    # 31. FINAL OUTPUT
    # ============================================================

    print(
        "\n" + "=" * 80
    )

    print(
        f"STEP 5 COMPLETED SUCCESSFULLY FOR "
        f"{network_name}"
    )

    print(
        "=" * 80
    )


    print(
        "\nSaved files:"
    )


    print(
        f"Adam DNN model         -> "
        f"{ADAM_MODEL_FILE}"
    )


    print(
        f"Quantum VQC model      -> "
        f"{QUANTUM_MODEL_FILE}"
    )


    print(
        f"Adam search results    -> "
        f"{ADAM_RESULTS_FILE}"
    )


    print(
        f"Quantum search results -> "
        f"{QUANTUM_RESULTS_FILE}"
    )


    # ============================================================
    # 32. RETURN RESULTS
    # ============================================================

    return {

        'network_name':
            network_name,

        'adam_model':
            best_adam_model,

        'quantum_model':
            best_vqc,

        'adam_best_config':
            best_adam_cfg,

        'quantum_best_config':
            best_q_cfg,

        'adam_model_file':
            ADAM_MODEL_FILE,

        'quantum_model_file':
            QUANTUM_MODEL_FILE,

        'adam_results_file':
            ADAM_RESULTS_FILE,

        'quantum_results_file':
            QUANTUM_RESULTS_FILE,

        'pca':
            pca,

        'pca_angle_scale':
            PCA_ANGLE_SCALE,

        'n_qubits':
            N_QUBITS

    }

In [12]:
# ================================================================
# RUN STEP 5 FOR ALL FOUR NETWORKS
# Adam-Optimized DNN + Quantum VQC
# ================================================================

print("\n")
print("#" * 100)
print("STARTING STEP 5 FOR ALL FOUR NETWORKS")
print("#" * 100)


STEP5_RESULTS = {}


for NETWORK_NAME in NETWORKS:

    print("\n")
    print("#" * 100)
    print(
        f"STARTING STEP 5 FOR NETWORK: "
        f"{NETWORK_NAME}"
    )
    print("#" * 100)


    # ------------------------------------------------------------
    # Get network-specific configuration
    # ------------------------------------------------------------

    NETWORK_CONFIG = NETWORK_CONFIGS[
        NETWORK_NAME
    ]


    # ------------------------------------------------------------
    # Get network-specific FLAT paths
    #
    # IMPORTANT:
    # Cell 1 creates:
    #
    # PATHS[NETWORK_NAME] = {
    #     'output_dir': ...,
    #     'splits_file': ...,
    #     'scaler_file': ...,
    #     ...
    # }
    #
    # Therefore pass:
    #
    # PATHS[NETWORK_NAME]
    #
    # NOT:
    #
    # PATHS[NETWORK_NAME]['paths']
    # ------------------------------------------------------------

    NETWORK_PATHS = PATHS[
        NETWORK_NAME
    ]


    # ------------------------------------------------------------
    # Run Step 5
    # ------------------------------------------------------------

    STEP5_RESULTS[
        NETWORK_NAME
    ] = run_step5_adam_quantum(

        network_name=NETWORK_NAME,

        config=NETWORK_CONFIG,

        paths=NETWORK_PATHS

    )


    print("\n")
    print("-" * 100)

    print(
        f"STEP 5 FINISHED FOR NETWORK: "
        f"{NETWORK_NAME}"
    )

    print("-" * 100)


# ================================================================
# STEP 5 COMPLETE FOR ALL NETWORKS
# ================================================================

print("\n")
print("#" * 100)
print("STEP 5 COMPLETED FOR ALL FOUR NETWORKS")
print("#" * 100)


print("\nCompleted networks:")


for NETWORK_NAME in NETWORKS:

    print(
        f"  [OK] {NETWORK_NAME}"
    )


print("\nSTEP5_RESULTS contains results for:")


print(
    list(
        STEP5_RESULTS.keys()
    )
)


print("\n" + "=" * 100)

print(
    "STEP 5 - ALL NETWORKS COMPLETE"
)

print("=" * 100)



####################################################################################################
STARTING STEP 5 FOR ALL FOUR NETWORKS
####################################################################################################


####################################################################################################
STARTING STEP 5 FOR NETWORK: YDIP
####################################################################################################


STEP 5: ADAM-OPTIMIZED DNN + QUANTUM VQC - YDIP

Network name : YDIP
Splits file  : /content/YDIP_results/splits.npz
Scaler file  : /content/YDIP_results/scaler.joblib
Output folder: /content/YDIP_results

----------------------------------------------------------------------
1. Loading network-specific train/validation/test splits
----------------------------------------------------------------------
Train samples      : 3330
Validation samples : 714
Test samples       : 714

------------------------------------

In [13]:
# ===============================================================
# STEP 6: FULL VALIDATION & COMPARISON (4 MODELS)
# ===============================================================
#
# Models:
#     1. Basic MLP
#     2. Advanced Deep NN
#     3. Adam-Optimized DNN
#     4. Quantum VQC
#
# Networks:
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# PRIMARY CONFIGURATION FROM CELL 1:
#
#     NETWORKS
#     NETWORK_CONFIGS
#     PATHS
#
# PATHS structure:
#
# PATHS[NETWORK_NAME] = {
#
#     'output_dir': ...,
#     'features_file': ...,
#     'splits_file': ...,
#     'scaler_file': ...,
#     'basic_model_file': ...,
#     'basic_search_file': ...,
#     'advanced_model_file': ...,
#     'advanced_search_file': ...,
#     'adam_model_file': ...,
#     'adam_search_file': ...,
#     'quantum_model_file': ...,
#     'quantum_search_file': ...,
#     'comparison_plot': ...,
#     'comparison_table': ...
#
# }
#
# COMPATIBILITY:
#
# The Cell 1 configuration also creates:
#
# NETWORK_PATHS[NETWORK_NAME] = {
#
#     'output_dir': ...,
#
#     'paths': PATHS[NETWORK_NAME]
#
# }
#
# Therefore this Step 6 primarily uses PATHS but can fall back
# to NETWORK_PATHS if required.
#
# ===============================================================


# ===============================================================
# 0. IMPORTS
# ===============================================================

import os
import sys
import numpy as np
import pandas as pd
import joblib
import pickle

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef
)

import matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt


# ===============================================================
# 1. RANDOM STATE
# ===============================================================

RANDOM_STATE = 42


# ===============================================================
# 2. CUSTOM MODULE PATH
# ===============================================================

WORK_DIR = '/content'

if WORK_DIR not in sys.path:

    sys.path.insert(
        0,
        WORK_DIR
    )


# ===============================================================
# 3. IMPORT CUSTOM MODEL CLASSES
#
# These modules must already exist in /content.
#
#     /content/deep_nn.py
#     /content/adam_plain_nn.py
#     /content/quantum_vqc.py
#
# ===============================================================

try:

    from deep_nn import DeepNN

except ImportError as e:

    raise ImportError(

        "Could not import DeepNN from deep_nn.py.\n"
        "Make sure deep_nn.py exists in /content "
        "and was created before running Step 6."

    ) from e


try:

    from adam_plain_nn import AdamPlainNN

except ImportError as e:

    raise ImportError(

        "Could not import AdamPlainNN from "
        "adam_plain_nn.py.\n"
        "Make sure adam_plain_nn.py exists in /content "
        "and was created before running Step 6."

    ) from e


try:

    from quantum_vqc import VQC

except ImportError as e:

    raise ImportError(

        "Could not import VQC from quantum_vqc.py.\n"
        "Make sure quantum_vqc.py exists in /content "
        "and was created before running Step 6."

    ) from e


# ===============================================================
# 4. VERIFY CELL 1 VARIABLES
#
# PRIMARY VARIABLES:
#
#     NETWORKS
#     NETWORK_CONFIGS
#     PATHS
#
# OPTIONAL COMPATIBILITY VARIABLE:
#
#     NETWORK_PATHS
#
# ===============================================================

if 'NETWORKS' not in globals():

    raise NameError(

        "NETWORKS is not defined.\n\n"
        "Run Cell 1 before running Step 6."

    )


if 'NETWORK_CONFIGS' not in globals():

    raise NameError(

        "NETWORK_CONFIGS is not defined.\n\n"
        "Run Cell 1 before running Step 6."

    )


if 'PATHS' not in globals():

    raise NameError(

        "PATHS is not defined.\n\n"
        "Run Cell 1 before running Step 6."

    )


# ===============================================================
# 5. REQUIRED NETWORKS
#
# Use NETWORKS from Cell 1 as the authoritative list.
#
# Expected:
#
#     YDIP
#     YMIPS
#     YMBD
#     YHQ
#
# ===============================================================

EXPECTED_NETWORKS = [

    'YDIP',

    'YMIPS',

    'YMBD',

    'YHQ'

]


# ===============================================================
# 6. VERIFY NETWORK LIST
# ===============================================================

print(

    "\nChecking network configuration..."

)


print(

    f"NETWORKS from Cell 1: "
    f"{NETWORKS}"

)


# ---------------------------------------------------------------
# Check expected networks
# ---------------------------------------------------------------

missing_from_networks = [

    network_name

    for network_name in EXPECTED_NETWORKS

    if network_name not in NETWORKS

]


if missing_from_networks:

    raise KeyError(

        "The following expected networks are missing "
        "from NETWORKS:\n\n"

        +

        "\n".join(

            f"  - {name}"

            for name in missing_from_networks

        )

    )


# ---------------------------------------------------------------
# Check NETWORK_CONFIGS
# ---------------------------------------------------------------

for network_name in EXPECTED_NETWORKS:

    if network_name not in NETWORK_CONFIGS:

        raise KeyError(

            f"Network '{network_name}' is missing "
            f"from NETWORK_CONFIGS."

        )


# ---------------------------------------------------------------
# Check PATHS
# ---------------------------------------------------------------

for network_name in EXPECTED_NETWORKS:

    if network_name not in PATHS:

        raise KeyError(

            f"Network '{network_name}' is missing "
            f"from PATHS."

        )


print(

    "\nNetwork configuration verified successfully."

)


# ===============================================================
# 7. REQUIRED PATH KEYS
#
# These are the keys created by Cell 1 -> create_paths().
#
# ===============================================================

REQUIRED_PATH_KEYS = [

    'output_dir',

    'splits_file',

    'scaler_file',

    'basic_model_file',

    'advanced_model_file',

    'adam_model_file',

    'quantum_model_file',

    'comparison_plot',

    'comparison_table'

]


# ===============================================================
# 8. SINGLE-NETWORK STEP 6 FUNCTION
# ===============================================================

def run_step6_validation(

    NETWORK_NAME,

    paths

):


    # ===========================================================
    # 8.1 VALIDATE PATH DICTIONARY
    # ===========================================================

    missing_path_keys = [

        key

        for key in REQUIRED_PATH_KEYS

        if key not in paths

    ]


    if missing_path_keys:

        raise KeyError(

            f"\n{NETWORK_NAME}: Missing path keys:\n"

            +

            "\n".join(

                f"  - {key}"

                for key in missing_path_keys

            )

            +

            "\n\n"
            "These paths should be generated by "
            "Cell 1 -> create_paths()."

        )


    # ===========================================================
    # 8.2 EXTRACT OUTPUT DIRECTORY
    #
    # output_dir is already stored inside PATHS.
    #
    # ===========================================================

    OUTPUT_DIR = paths[

        'output_dir'

    ]


    # ===========================================================
    # 8.3 CREATE OUTPUT DIRECTORY
    # ===========================================================

    os.makedirs(

        OUTPUT_DIR,

        exist_ok=True

    )


    # ===========================================================
    # 8.4 DISPLAY NETWORK
    # ===========================================================

    print(
        "\n\n"
    )


    print(
        "=" * 100
    )


    print(

        f"STEP 6: FULL VALIDATION & COMPARISON - "
        f"{NETWORK_NAME}"

    )


    print(
        "=" * 100
    )


    print(

        f"\nCurrent network : "
        f"{NETWORK_NAME}"

    )


    print(

        f"Output folder   : "
        f"{OUTPUT_DIR}"

    )


    # ===========================================================
    # 8.5 REQUIRED FILES
    #
    # Use the exact paths generated by Cell 1.
    #
    # ===========================================================

    files_to_check = {

        'splits':
            paths['splits_file'],

        'scaler':
            paths['scaler_file'],

        'basic_model':
            paths['basic_model_file'],

        'advanced_model':
            paths['advanced_model_file'],

        'adam_model':
            paths['adam_model_file'],

        'quantum_model':
            paths['quantum_model_file']

    }


    print(

        "\nModel and data files:"

    )


    for name, filepath in files_to_check.items():

        print(

            f"{name:18s}: "
            f"{filepath}"

        )


    # ===========================================================
    # 8.6 CHECK FILE EXISTENCE
    # ===========================================================

    print(

        "\nChecking required files..."

    )


    missing_files = []

    existing_files = []


    for name, filepath in files_to_check.items():

        if os.path.isfile(

            filepath

        ):

            existing_files.append(

                name

            )


            print(

                f"  [OK]      {name}: "
                f"{filepath}"

            )

        else:

            missing_files.append(

                name

            )


            print(

                f"  [MISSING] {name}: "
                f"{filepath}"

            )


    # ===========================================================
    # 8.7 STOP IF FILES ARE MISSING
    # ===========================================================

    if missing_files:

        raise FileNotFoundError(

            "\n\nThe following required files are missing "
            f"for {NETWORK_NAME}:\n\n"

            +

            "\n".join(

                f"  - {name}: "
                f"{files_to_check[name]}"

                for name in missing_files

            )

            +

            "\n\n"
            "Step 6 cannot run until these files exist.\n\n"

            "Required sequence:\n"

            "  Step 1 -> Feature generation\n"

            "  Step 2 -> Basic MLP + scaler + splits\n"

            "  Step 3 -> Advanced Deep NN\n"

            "  Step 4 -> Existing pipeline step\n"

            "  Step 5 -> Adam DNN + Quantum VQC\n"

            "  Step 6 -> Validation and comparison\n"

        )


    print(

        "\nAll required Step 6 files found."

    )


    # ===========================================================
    # 8.8 LOAD SPLITS
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "1. Loading train / validation / test splits"

    )


    print(

        "-" * 70

    )


    splits = np.load(

        paths['splits_file']

    )


    # -----------------------------------------------------------
    # Validate split keys
    # -----------------------------------------------------------

    required_split_keys = [

        'X_train',

        'y_train',

        'X_val',

        'y_val',

        'X_test',

        'y_test'

    ]


    missing_split_keys = [

        key

        for key in required_split_keys

        if key not in splits

    ]


    if missing_split_keys:

        raise KeyError(

            f"{NETWORK_NAME}: splits.npz is missing keys: "

            +

            ", ".join(

                missing_split_keys

            )

        )


    X_train = splits['X_train']

    y_train = splits['y_train']


    X_val = splits['X_val']

    y_val = splits['y_val']


    X_test = splits['X_test']

    y_test = splits['y_test']


    print(

        f"Train samples      : "
        f"{len(X_train)}"

    )


    print(

        f"Validation samples : "
        f"{len(X_val)}"

    )


    print(

        f"Test samples       : "
        f"{len(X_test)}"

    )


    # ===========================================================
    # 8.9 CHECK LABELS
    # ===========================================================

    print(

        "\nLabel distribution:"

    )


    for split_name, labels in [

        ('Train', y_train),

        ('Validation', y_val),

        ('Test', y_test)

    ]:


        unique, counts = np.unique(

            labels,

            return_counts=True

        )


        print(

            f"\n{split_name}:"

        )


        for label, count in zip(

            unique,

            counts

        ):


            class_name = (

                'essential'

                if int(label) == 1

                else 'non-essential'

            )


            print(

                f"  {class_name}: "
                f"{count}"

            )


    # ===========================================================
    # 8.10 LOAD SCALER
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "2. Loading StandardScaler"

    )


    print(

        "-" * 70

    )


    scaler = joblib.load(

        paths['scaler_file']

    )


    X_val_s = scaler.transform(

        X_val

    )


    X_test_s = scaler.transform(

        X_test

    )


    print(

        "StandardScaler loaded successfully."

    )


    print(

        f"Scaled validation shape: "
        f"{X_val_s.shape}"

    )


    print(

        f"Scaled test shape      : "
        f"{X_test_s.shape}"

    )


    # ===========================================================
    # 8.11 LOAD BASIC MLP
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "3. Loading Basic MLP"

    )


    print(

        "-" * 70

    )


    basic_model = joblib.load(

        paths['basic_model_file']

    )


    print(

        "Basic MLP loaded successfully."

    )


    # ===========================================================
    # 8.12 LOAD ADVANCED MODEL
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "4. Loading Advanced Deep NN"

    )


    print(

        "-" * 70

    )


    with open(

        paths['advanced_model_file'],

        'rb'

    ) as f:

        advanced_bundle = pickle.load(

            f

        )


    if (

        isinstance(

            advanced_bundle,

            dict

        )

        and

        'model' in advanced_bundle

    ):

        advanced_model = advanced_bundle[

            'model'

        ]

    else:

        advanced_model = advanced_bundle


    print(

        "Advanced Deep NN loaded successfully."

    )


    # ===========================================================
    # 8.13 LOAD ADAM MODEL
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "5. Loading Adam-Optimized DNN"

    )


    print(

        "-" * 70

    )


    with open(

        paths['adam_model_file'],

        'rb'

    ) as f:

        adam_bundle = pickle.load(

            f

        )


    if (

        isinstance(

            adam_bundle,

            dict

        )

        and

        'model' in adam_bundle

    ):

        adam_model = adam_bundle[

            'model'

        ]

    else:

        adam_model = adam_bundle


    print(

        "Adam-Optimized DNN loaded successfully."

    )


    # ===========================================================
    # 8.14 LOAD QUANTUM MODEL
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "6. Loading Quantum VQC"

    )


    print(

        "-" * 70

    )


    with open(

        paths['quantum_model_file'],

        'rb'

    ) as f:

        quantum_bundle = pickle.load(

            f

        )


    if not isinstance(

        quantum_bundle,

        dict

    ):

        raise ValueError(

            "Quantum VQC file must contain a dictionary."

        )


    required_quantum_keys = [

        'model',

        'pca',

        'pca_angle_scale'

    ]


    missing_quantum_keys = [

        key

        for key in required_quantum_keys

        if key not in quantum_bundle

    ]


    if missing_quantum_keys:

        raise KeyError(

            "Quantum VQC bundle is missing required keys: "

            +

            ", ".join(

                missing_quantum_keys

            )

        )


    vqc_model = quantum_bundle[

        'model'

    ]


    pca = quantum_bundle[

        'pca'

    ]


    PCA_ANGLE_SCALE = quantum_bundle[

        'pca_angle_scale'

    ]


    n_qubits = quantum_bundle.get(

        'n_qubits',

        pca.n_components_

    )


    print(

        "Quantum VQC loaded successfully."

    )


    print(

        f"Quantum qubits       : "
        f"{n_qubits}"

    )


    print(

        f"Quantum PCA features : "
        f"{pca.n_components_}"

    )


    # ===========================================================
    # 8.15 PREPARE QUANTUM DATA
    # ===========================================================

    print(

        "\n" + "-" * 70

    )


    print(

        "7. Preparing Quantum VQC input"

    )


    print(

        "-" * 70

    )


    def to_angles(

        X

    ):


        Xp = pca.transform(

            X

        )


        Xp = (

            Xp /

            PCA_ANGLE_SCALE

        )


        Xp = np.clip(

            Xp,

            -1.0,

            1.0

        )


        return (

            Xp * np.pi

        )


    Xq_val_s = to_angles(

        X_val_s

    )


    Xq_test_s = to_angles(

        X_test_s

    )


    print(

        f"Quantum validation shape: "
        f"{Xq_val_s.shape}"

    )


    print(

        f"Quantum test shape      : "
        f"{Xq_test_s.shape}"

    )


    # ===========================================================
    # 8.16 PROBABILITY EXTRACTION
    # ===========================================================

    def get_positive_class_probability(

        model,

        X

    ):


        if not hasattr(

            model,

            'predict_proba'

        ):

            raise AttributeError(

                f"Model of type "
                f"{type(model).__name__} "
                f"does not provide predict_proba()."

            )


        proba = model.predict_proba(

            X

        )


        proba = np.asarray(

            proba

        )


        if proba.ndim == 1:

            return proba.reshape(

                -1

            )


        if (

            proba.ndim == 2

            and

            proba.shape[1] == 1

        ):

            return proba[:, 0]


        if (

            proba.ndim == 2

            and

            proba.shape[1] == 2

        ):

            return proba[:, 1]


        raise ValueError(

            f"Unexpected predict_proba output shape: "
            f"{proba.shape}"

        )


    # ===========================================================
    # 8.17 BEST VALIDATION THRESHOLD
    # ===========================================================

    def best_threshold(

        y_true,

        proba

    ):


        fpr, tpr, thresholds = roc_curve(

            y_true,

            proba

        )


        j_scores = (

            tpr -

            fpr

        )


        best_idx = np.argmax(

            j_scores

        )


        threshold = thresholds[

            best_idx

        ]


        finite_thresholds = thresholds[

            np.isfinite(

                thresholds

            )

        ]


        if not np.isfinite(

            threshold

        ):


            if len(

                finite_thresholds

            ) > 0:


                threshold = finite_thresholds[

                    0

                ]


            else:

                threshold = 0.5


        return float(

            np.clip(

                threshold,

                0.0,

                1.0

            )

        )


    # ===========================================================
    # 8.18 FULL EVALUATION
    # ===========================================================

    def full_eval(

        y_true,

        proba,

        threshold

    ):


        pred = (

            proba >= threshold

        ).astype(

            int

        )


        metrics = {

            'threshold':

                float(

                    threshold

                ),

            'accuracy':

                accuracy_score(

                    y_true,

                    pred

                ),

            'precision':

                precision_score(

                    y_true,

                    pred,

                    zero_division=0

                ),

            'recall':

                recall_score(

                    y_true,

                    pred,

                    zero_division=0

                ),

            'f1':

                f1_score(

                    y_true,

                    pred,

                    zero_division=0

                ),

            'mcc':

                matthews_corrcoef(

                    y_true,

                    pred

                ),

            'roc_auc':

                roc_auc_score(

                    y_true,

                    proba

                )

        }


        cm = confusion_matrix(

            y_true,

            pred

        )


        return (

            metrics,

            cm

        )


    # ===========================================================
    # 8.19 MODEL REGISTRY
    # ===========================================================

    model_registry = [

        (

            'Basic MLP',

            basic_model,

            X_val_s,

            X_test_s

        ),

        (

            'Advanced Deep NN',

            advanced_model,

            X_val_s,

            X_test_s

        ),

        (

            'Adam-Optimized DNN',

            adam_model,

            X_val_s,

            X_test_s

        ),

        (

            'Quantum VQC',

            vqc_model,

            Xq_val_s,

            Xq_test_s

        )

    ]


    # ===========================================================
    # 8.20 INITIALIZE
    # ===========================================================

    results = {}


    fig, axes = plt.subplots(

        1,

        2,

        figsize=(12, 5)

    )


    ax_roc = axes[0]

    ax_pr = axes[1]


    # ===========================================================
    # 8.21 EVALUATE ALL MODELS
    # ===========================================================

    for (

        name,

        model,

        Xv,

        Xt

    ) in model_registry:


        print(

            "\n" + "=" * 70

        )


        print(

            f"EVALUATING: {name}"

        )


        print(

            "=" * 70

        )


        # -------------------------------------------------------
        # VALIDATION PROBABILITY
        # -------------------------------------------------------

        val_proba = get_positive_class_probability(

            model,

            Xv

        )


        # -------------------------------------------------------
        # TEST PROBABILITY
        # -------------------------------------------------------

        test_proba = get_positive_class_probability(

            model,

            Xt

        )


        # -------------------------------------------------------
        # VALIDATE LENGTHS
        # -------------------------------------------------------

        if len(

            val_proba

        ) != len(

            y_val

        ):


            raise ValueError(

                f"{name}: Validation prediction length "
                f"{len(val_proba)} != "
                f"{len(y_val)}"

            )


        if len(

            test_proba

        ) != len(

            y_test

        ):


            raise ValueError(

                f"{name}: Test prediction length "
                f"{len(test_proba)} != "
                f"{len(y_test)}"

            )


        # -------------------------------------------------------
        # VALIDATION-BASED THRESHOLD
        #
        # Threshold is determined ONLY using validation data.
        #
        # The test set is never used to select the threshold.
        #
        # -------------------------------------------------------

        thr = best_threshold(

            y_val,

            val_proba

        )


        # -------------------------------------------------------
        # DEFAULT THRESHOLD = 0.5
        # -------------------------------------------------------

        metrics_default, cm_default = full_eval(

            y_test,

            test_proba,

            0.5

        )


        # -------------------------------------------------------
        # VALIDATION-TUNED THRESHOLD
        # -------------------------------------------------------

        metrics_tuned, cm_tuned = full_eval(

            y_test,

            test_proba,

            thr

        )


        # -------------------------------------------------------
        # STORE RESULTS
        # -------------------------------------------------------

        results[name] = {

            'default':
                metrics_default,

            'tuned':
                metrics_tuned,

            'cm_default':
                cm_default,

            'cm_tuned':
                cm_tuned,

            'thr':
                thr

        }


        # -------------------------------------------------------
        # ROC CURVE
        # -------------------------------------------------------

        fpr, tpr, _ = roc_curve(

            y_test,

            test_proba

        )


        ax_roc.plot(

            fpr,

            tpr,

            label=(

                f"{name} "
                f"(AUC="
                f"{metrics_default['roc_auc']:.3f})"

            )

        )


        # -------------------------------------------------------
        # PRECISION-RECALL CURVE
        # -------------------------------------------------------

        precision, recall, _ = precision_recall_curve(

            y_test,

            test_proba

        )


        ax_pr.plot(

            recall,

            precision,

            label=name

        )


        # -------------------------------------------------------
        # PRINT RESULTS
        # -------------------------------------------------------

        print(

            f"\nOptimal Youden's J threshold: "
            f"{thr:.4f}"

        )


        print(

            "\n-- Test set @ threshold 0.5 --"

        )


        for k, v in metrics_default.items():

            print(

                f"  {k:>10s}: "
                f"{v:.4f}"

            )


        print(

            "\nConfusion matrix:"

        )


        print(

            cm_default

        )


        print(

            f"\n-- Test set @ tuned threshold "
            f"{thr:.4f} --"

        )


        for k, v in metrics_tuned.items():

            print(

                f"  {k:>10s}: "
                f"{v:.4f}"

            )


        print(

            "\nConfusion matrix:"

        )


        print(

            cm_tuned

        )


    # ===========================================================
    # 8.22 FORMAT ROC
    # ===========================================================

    ax_roc.plot(

        [0, 1],

        [0, 1],

        'k--',

        alpha=0.3

    )


    ax_roc.set_xlabel(

        'False Positive Rate'

    )


    ax_roc.set_ylabel(

        'True Positive Rate'

    )


    ax_roc.set_title(

        f'ROC Curve — {NETWORK_NAME} Test Set'

    )


    ax_roc.legend(

        fontsize=8

    )


    # ===========================================================
    # 8.23 FORMAT PR
    # ===========================================================

    ax_pr.set_xlabel(

        'Recall'

    )


    ax_pr.set_ylabel(

        'Precision'

    )


    ax_pr.set_title(

        f'Precision-Recall Curve — {NETWORK_NAME} Test Set'

    )


    ax_pr.legend(

        fontsize=8

    )


    # ===========================================================
    # 8.24 SAVE PLOT
    #
    # Use the exact comparison_plot path from Cell 1.
    #
    # ===========================================================

    fig.tight_layout()


    PLOT_FILE = paths[

        'comparison_plot'

    ]


    fig.savefig(

        PLOT_FILE,

        dpi=150,

        bbox_inches='tight'

    )


    plt.close(

        fig

    )


    print(

        f"\nSaved plot -> "
        f"{PLOT_FILE}"

    )


    # ===========================================================
    # 8.25 CREATE SUMMARY TABLE
    # ===========================================================

    rows = []


    for name, res in results.items():


        rows.append(

            {

                'network':
                    NETWORK_NAME,

                'model':
                    name,

                'threshold_setting':
                    'default_threshold(0.5)',

                **res['default']

            }

        )


        rows.append(

            {

                'network':
                    NETWORK_NAME,

                'model':
                    name,

                'threshold_setting':
                    f"tuned_threshold({res['thr']:.3f})",

                **res['tuned']

            }

        )


    summary = pd.DataFrame(

        rows

    )


    # ===========================================================
    # 8.26 SAVE CSV
    #
    # Use the exact comparison_table path from Cell 1.
    #
    # ===========================================================

    FINAL_COMPARISON_FILE = paths[

        'comparison_table'

    ]


    summary.to_csv(

        FINAL_COMPARISON_FILE,

        index=False

    )


    # ===========================================================
    # 8.27 PRINT SUMMARY
    # ===========================================================

    print(

        "\n" + "=" * 70

    )


    print(

        f"FINAL 4-MODEL COMPARISON - "
        f"{NETWORK_NAME}"

    )


    print(

        "=" * 70

    )


    print(

        summary.round(

            4

        ).to_string(

            index=False

        )

    )


    print(

        f"\nSaved final comparison table -> "
        f"{FINAL_COMPARISON_FILE}"

    )


    print(

        "\n" + "=" * 70

    )


    print(

        f"STEP 6 COMPLETED SUCCESSFULLY "
        f"FOR {NETWORK_NAME}"

    )


    print(

        "=" * 70

    )


    # ===========================================================
    # RETURN
    # ===========================================================

    return {

        'network_name':
            NETWORK_NAME,

        'results':
            results,

        'summary':
            summary,

        'plot_file':
            PLOT_FILE,

        'comparison_file':
            FINAL_COMPARISON_FILE,

        'output_dir':
            OUTPUT_DIR

    }


# ===============================================================
# 9. RUN STEP 6 FOR ALL NETWORKS
#
# PRIMARY:
#
#     NETWORKS
#     PATHS
#
# No hard-coded output directories are created here.
#
# ===============================================================

STEP6_RESULTS = {}

STEP6_STATUS = {}


print(

    "\n\n"

)


print(

    "#" * 100

)


print(

    "STARTING STEP 6 FOR ALL FOUR NETWORKS"

)


print(

    "#" * 100

)


for NETWORK_NAME in EXPECTED_NETWORKS:


    # ===========================================================
    # CHECK NETWORK IN PATHS
    # ===========================================================

    if NETWORK_NAME not in PATHS:


        STEP6_STATUS[

            NETWORK_NAME

        ] = (

            'SKIPPED: Missing PATHS entry'

        )


        print(

            f"\n[SKIPPED] {NETWORK_NAME}: "
            f"Missing PATHS entry."

        )


        continue


    # ===========================================================
    # GET FLAT PATH DICTIONARY
    #
    # Example:
    #
    # PATHS['YDIP']['splits_file']
    #
    # ===========================================================

    paths = PATHS[

        NETWORK_NAME

    ]


    # ===========================================================
    # RUN STEP 6
    # ===========================================================

    print(

        "\n\n"

    )


    print(

        "#" * 100

    )


    print(

        f"STARTING STEP 6 FOR NETWORK: "
        f"{NETWORK_NAME}"

    )


    print(

        "#" * 100

    )


    try:


        STEP6_RESULTS[

            NETWORK_NAME

        ] = run_step6_validation(

            NETWORK_NAME=NETWORK_NAME,

            paths=paths

        )


        STEP6_STATUS[

            NETWORK_NAME

        ] = (

            'COMPLETED SUCCESSFULLY'

        )


    except Exception as e:


        STEP6_STATUS[

            NETWORK_NAME

        ] = (

            f'FAILED: {type(e).__name__}: {e}'

        )


        print(

            "\n" + "!" * 100

        )


        print(

            f"STEP 6 FAILED FOR {NETWORK_NAME}"

        )


        print(

            f"Error type: "
            f"{type(e).__name__}"

        )


        print(

            f"Error: "
            f"{e}"

        )


        print(

            "!" * 100

        )


        # Continue to next network

        continue


# ===============================================================
# 10. FINAL MULTI-NETWORK SUMMARY
# ===============================================================

print(

    "\n\n"

)


print(

    "=" * 100

)


print(

    "STEP 6 MULTI-NETWORK EXECUTION SUMMARY"

)


print(

    "=" * 100

)


for NETWORK_NAME in EXPECTED_NETWORKS:


    print(

        f"\n{NETWORK_NAME}:"

    )


    print(

        f"  Status: "
        f"{STEP6_STATUS.get(NETWORK_NAME, 'NOT RUN')}"

    )


    if NETWORK_NAME in STEP6_RESULTS:


        result = STEP6_RESULTS[

            NETWORK_NAME

        ]


        print(

            f"  Output directory:"
            f"\n    {result['output_dir']}"

        )


        print(

            f"  Comparison plot:"
            f"\n    {result['plot_file']}"

        )


        print(

            f"  Comparison table:"
            f"\n    {result['comparison_file']}"

        )


# ===============================================================
# 11. FINAL MESSAGE
# ===============================================================

successful_networks = [

    name

    for name in EXPECTED_NETWORKS

    if name in STEP6_RESULTS

]


failed_networks = [

    name

    for name in EXPECTED_NETWORKS

    if name not in STEP6_RESULTS

]


print(

    "\n" + "=" * 100

)


print(

    "STEP 6 FINISHED"

)


print(

    "=" * 100

)


print(

    f"\nSuccessfully validated: "
    f"{len(successful_networks)} / "
    f"{len(EXPECTED_NETWORKS)}"

)


if successful_networks:


    print(

        "\nSuccessful networks:"

    )


    for name in successful_networks:


        print(

            f"  [OK] {name}"

        )


if failed_networks:


    print(

        "\nNetworks requiring attention:"

    )


    for name in failed_networks:


        print(

            f"  [FAILED/SKIPPED] {name}"

        )


print(

    "\n" + "=" * 100

)


Checking network configuration...
NETWORKS from Cell 1: ['YDIP', 'YMIPS', 'YMBD', 'YHQ']

Network configuration verified successfully.



####################################################################################################
STARTING STEP 6 FOR ALL FOUR NETWORKS
####################################################################################################



####################################################################################################
STARTING STEP 6 FOR NETWORK: YDIP
####################################################################################################



STEP 6: FULL VALIDATION & COMPARISON - YDIP

Current network : YDIP
Output folder   : /content/YDIP_results

Model and data files:
splits            : /content/YDIP_results/splits.npz
scaler            : /content/YDIP_results/scaler.joblib
basic_model       : /content/YDIP_results/basic_model.joblib
advanced_model    : /content/YDIP_results/advanced_model.pkl
adam_model       

In [14]:
import numpy as np
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

class DeepNN:
    def __init__(self, layer_sizes, dropout=0.2, l2=1e-4, lr=1e-3,
                 class_weight=1.0, seed=RANDOM_STATE):
        self.layer_sizes = layer_sizes
        self.dropout = dropout
        self.l2 = l2
        self.lr = lr
        self.class_weight = class_weight  # weight applied to the positive (essential) class
        rng = np.random.RandomState(seed)
        self.params = {}
        self.bn = {}
        L = len(layer_sizes) - 1
        self.L = L
        for l in range(1, L + 1):
            fan_in = layer_sizes[l - 1]
            self.params[f'W{l}'] = rng.randn(layer_sizes[l - 1], layer_sizes[l]) * np.sqrt(2.0 / fan_in)
            self.params[f'b{l}'] = np.zeros((1, layer_sizes[l]))
            if l < L:  # batch-norm on hidden layers only
                self.bn[f'gamma{l}'] = np.ones((1, layer_sizes[l]))
                self.bn[f'beta{l}']  = np.zeros((1, layer_sizes[l]))
                self.bn[f'run_mean{l}'] = np.zeros((1, layer_sizes[l]))
                self.bn[f'run_var{l}']  = np.ones((1, layer_sizes[l]))
        # Adam moments
        self.m = {k: np.zeros_like(v) for k, v in {**self.params, **self.bn}.items()
                   if not k.startswith('run_')}
        self.v = {k: np.zeros_like(v) for k, v in {**self.params, **self.bn}.items()
                   if not k.startswith('run_')}
        self.t = 0

    @staticmethod
    def relu(z):
        return np.maximum(0, z)

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

    def forward(self, X, training=True):
        cache = {'A0': X}
        A = X
        for l in range(1, self.L + 1):
            Z = A @ self.params[f'W{l}'] + self.params[f'b{l}']
            if l < self.L:
                # batch norm
                if training:
                    mu = Z.mean(axis=0, keepdims=True)
                    var = Z.var(axis=0, keepdims=True)
                    self.bn[f'run_mean{l}'] = 0.9 * self.bn[f'run_mean{l}'] + 0.1 * mu
                    self.bn[f'run_var{l}']  = 0.9 * self.bn[f'run_var{l}']  + 0.1 * var
                else:
                    mu, var = self.bn[f'run_mean{l}'], self.bn[f'run_var{l}']
                Zn = (Z - mu) / np.sqrt(var + 1e-8)
                Zbn = self.bn[f'gamma{l}'] * Zn + self.bn[f'beta{l}']
                A = self.relu(Zbn)
                if training and self.dropout > 0:
                    mask = (np.random.rand(*A.shape) > self.dropout) / (1 - self.dropout)
                    A = A * mask
                    cache[f'mask{l}'] = mask
                cache[f'Z{l}'] = Z; cache[f'Zn{l}'] = Zn; cache[f'mu{l}'] = mu; cache[f'var{l}'] = var
            else:
                A = self.sigmoid(Z)
                cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A
        return A, cache

    def compute_loss(self, y_hat, y):
        y = y.reshape(-1, 1)
        w = np.where(y == 1, self.class_weight, 1.0)
        eps = 1e-8
        bce = -(w * (y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps)))
        data_loss = bce.mean()
        l2_loss = sum(np.sum(self.params[f'W{l}'] ** 2) for l in range(1, self.L + 1))
        return data_loss + self.l2 * l2_loss

    def backward(self, cache, y):
        m = y.shape[0]
        y = y.reshape(-1, 1)
        w = np.where(y == 1, self.class_weight, 1.0)
        grads = {}
        A_L = cache[f'A{self.L}']
        dZ = w * (A_L - y) / m
        for l in range(self.L, 0, -1):
            A_prev = cache[f'A{l-1}']
            grads[f'W{l}'] = A_prev.T @ dZ + 2 * self.l2 * self.params[f'W{l}']
            grads[f'b{l}'] = dZ.sum(axis=0, keepdims=True)
            if l > 1:
                dA_prev = dZ @ self.params[f'W{l}'].T
                lp = l - 1
                if self.dropout > 0 and f'mask{lp}' in cache:
                    dA_prev = dA_prev * cache[f'mask{lp}']
                dZbn = dA_prev * (cache[f'Zn{lp}'] * 0 + (cache[f'A{lp}'] > 0))  # relu' via post-dropout sign proxy
                # proper relu derivative uses pre-dropout activation sign; recompute cleanly:
                relu_mask = (cache[f'Zn{lp}'] * self.bn[f'gamma{lp}'] + self.bn[f'beta{lp}']) > 0
                dZbn = dA_prev * relu_mask
                grads[f'gamma{lp}'] = (dZbn * cache[f'Zn{lp}']).sum(axis=0, keepdims=True)
                grads[f'beta{lp}']  = dZbn.sum(axis=0, keepdims=True)
                std_inv = 1.0 / np.sqrt(cache[f'var{lp}'] + 1e-8)
                dZn = dZbn * self.bn[f'gamma{lp}']
                N = m
                dZ_prev = (1.0 / N) * std_inv * (
                    N * dZn - dZn.sum(axis=0, keepdims=True)
                    - cache[f'Zn{lp}'] * (dZn * cache[f'Zn{lp}']).sum(axis=0, keepdims=True)
                )
                dZ = dZ_prev
        return grads

    def step(self, grads, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        all_params = {**self.params, **self.bn}
        for k in grads:
            if k not in self.m:
                continue
            self.m[k] = beta1 * self.m[k] + (1 - beta1) * grads[k]
            self.v[k] = beta2 * self.v[k] + (1 - beta2) * (grads[k] ** 2)
            m_hat = self.m[k] / (1 - beta1 ** self.t)
            v_hat = self.v[k] / (1 - beta2 ** self.t)
            update = self.lr * m_hat / (np.sqrt(v_hat) + eps)
            if k in self.params:
                self.params[k] -= update
            else:
                self.bn[k] -= update

    def fit(self, X, y, X_val=None, y_val=None, epochs=150, batch_size=64,
             patience=15, verbose=False):
        n = X.shape[0]
        best_val_auc = -1
        best_state = None
        no_improve = 0
        history = []
        for epoch in range(epochs):
            perm = np.random.permutation(n)
            Xs, ys = X[perm], y[perm]
            for i in range(0, n, batch_size):
                xb = Xs[i:i + batch_size]
                yb = ys[i:i + batch_size]
                y_hat, cache = self.forward(xb, training=True)
                grads = self.backward(cache, yb)
                self.step(grads)
            train_pred, _ = self.forward(X, training=False)
            train_loss = self.compute_loss(train_pred, y)
            if X_val is not None:
                val_pred, _ = self.forward(X_val, training=False)
                val_auc = roc_auc_score(y_val, val_pred.ravel())
                history.append((epoch, train_loss, val_auc))
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    best_state = {k: v.copy() for k, v in {**self.params, **self.bn}.items()}
                    no_improve = 0
                else:
                    no_improve += 1
                if verbose and epoch % 20 == 0:
                    print(f"  epoch {epoch:3d}  train_loss={train_loss:.4f}  val_auc={val_auc:.4f}")
                if no_improve >= patience:
                    break
        if best_state is not None:
            for k, v in best_state.items():
                if k in self.params:
                    self.params[k] = v
                else:
                    self.bn[k] = v
        return history

    def predict_proba(self, X):
        p, _ = self.forward(X, training=False)
        return p.ravel()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)



In [15]:
"""
quantum_vqc.py
A from-scratch statevector simulator for a small variational quantum circuit
(no qiskit/pennylane available in this environment / no internet access to
install them), plus a Variational Quantum Classifier (VQC) trained with the
Adam optimizer using the parameter-shift rule for exact gradients.

Circuit design (n_qubits, e.g. 4):
  1. Angle encoding:  RY(x_i) on qubit i  for each of the n_qubits input features
  2. Variational layers (repeated `depth` times):
       - RY(theta) single-qubit rotations on every qubit
       - a ring of CNOT entangling gates (0->1->2->...->n-1->0)
  3. Readout: expectation value <Z> on qubit 0 -> mapped to a probability via
     a sigmoid, i.e.  p(essential) = sigmoid(scale * <Z_0>)

Gradients of every rotation angle (encoding is fixed/data, variational
angles are trainable) are computed with the parameter-shift rule:
    d<Z>/dtheta = ( <Z>(theta + pi/2) - <Z>(theta - pi/2) ) / 2
which is EXACT for Pauli rotation gates, then combined with the chain rule
through the sigmoid + binary cross-entropy loss, and used to update the
mean parameters via the Adam optimizer.
"""
import numpy as np
from sklearn.metrics import roc_auc_score


# ----------------------------------------------------------------------
# Statevector simulator primitives
# ----------------------------------------------------------------------
def ry_matrix(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]])


def apply_single_qubit_gate(state, gate, qubit, n_qubits):
    state = state.reshape([2] * n_qubits)
    state = np.moveaxis(state, qubit, 0)
    state = np.tensordot(gate, state, axes=([1], [0]))
    state = np.moveaxis(state, 0, qubit)
    return state.reshape(-1)


def apply_cnot(state, control, target, n_qubits):
    state = state.reshape([2] * n_qubits)
    state = np.moveaxis(state, [control, target], [0, 1])
    out = state.copy()
    out[1] = state[1, ::-1]           # flip target when control == 1
    out = np.moveaxis(out, [0, 1], [control, target])
    return out.reshape(-1)


def expectation_z(state, qubit, n_qubits):
    probs = np.abs(state) ** 2
    probs = probs.reshape([2] * n_qubits)
    p0 = probs.take(0, axis=qubit).sum()
    p1 = probs.take(1, axis=qubit).sum()
    return p0 - p1


# ----------------------------------------------------------------------
# Variational Quantum Classifier
# ----------------------------------------------------------------------
class VQC:
    def __init__(self, n_qubits=4, depth=2, scale=2.0, lr=0.05, seed=42):
        self.n_qubits = n_qubits
        self.depth = depth
        self.scale = scale
        self.lr = lr
        rng = np.random.RandomState(seed)
        # one trainable RY angle per qubit per layer
        self.theta = rng.uniform(-0.1, 0.1, size=(depth, n_qubits))
        # Adam state
        self.m = np.zeros_like(self.theta)
        self.v = np.zeros_like(self.theta)
        self.t = 0

    def _circuit(self, x, theta):
        """Run the encoding + variational circuit for one sample; return <Z_0>."""
        n = self.n_qubits
        state = np.zeros(2 ** n, dtype=complex)
        state[0] = 1.0
        # angle encoding
        for q in range(n):
            state = apply_single_qubit_gate(state, ry_matrix(x[q]), q, n)
        # variational layers
        for l in range(self.depth):
            for q in range(n):
                state = apply_single_qubit_gate(state, ry_matrix(theta[l, q]), q, n)
            for q in range(n - 1):
                state = apply_cnot(state, q, q + 1, n)
            state = apply_cnot(state, n - 1, 0, n)  # close the entangling ring
        return expectation_z(state, 0, n)

    def _predict_expZ(self, X, theta=None):
        theta = self.theta if theta is None else theta
        return np.array([self._circuit(x, theta) for x in X])

    def predict_proba(self, X):
        expz = self._predict_expZ(X)
        return 1.0 / (1.0 + np.exp(-self.scale * expz))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def _param_shift_grad(self, X, y, class_weight):
        """Exact parameter-shift gradient of the BCE loss w.r.t. every theta,
        combined with an Adam update."""
        shift = np.pi / 2
        grad = np.zeros_like(self.theta)
        base_expz = self._predict_expZ(X)
        proba = 1.0 / (1.0 + np.exp(-self.scale * base_expz))
        w = np.where(y == 1, class_weight, 1.0)
        # dLoss/dExpZ  (chain rule through sigmoid + weighted BCE)
        dL_dp = w * (proba - y) / len(y)          # d(BCE)/d(sigmoid input) simplifies to (p - y)
        dL_dexpz = dL_dp * self.scale              # sigmoid'(z)*scale folded via standard BCE+sigmoid identity

        for l in range(self.depth):
            for q in range(self.n_qubits):
                theta_plus = self.theta.copy(); theta_plus[l, q] += shift
                theta_minus = self.theta.copy(); theta_minus[l, q] -= shift
                expz_plus = self._predict_expZ(X, theta_plus)
                expz_minus = self._predict_expZ(X, theta_minus)
                dexpz_dtheta = (expz_plus - expz_minus) / 2.0
                grad[l, q] = np.sum(dL_dexpz * dexpz_dtheta)
        return grad

    def fit(self, X, y, X_val=None, y_val=None, epochs=25, batch_size=32,
            class_weight=1.0, patience=6, verbose=False):
        n = X.shape[0]
        best_val_auc, best_theta, no_improve = -1, self.theta.copy(), 0
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        for epoch in range(epochs):
            perm = np.random.permutation(n)
            Xs, ys = X[perm], y[perm]
            for i in range(0, n, batch_size):
                xb, yb = Xs[i:i + batch_size], ys[i:i + batch_size]
                grad = self._param_shift_grad(xb, yb, class_weight)
                self.t += 1
                self.m = beta1 * self.m + (1 - beta1) * grad
                self.v = beta2 * self.v + (1 - beta2) * (grad ** 2)
                m_hat = self.m / (1 - beta1 ** self.t)
                v_hat = self.v / (1 - beta2 ** self.t)
                self.theta -= self.lr * m_hat / (np.sqrt(v_hat) + eps)
            if X_val is not None:
                val_auc = roc_auc_score(y_val, self.predict_proba(X_val))
                if val_auc > best_val_auc:
                    best_val_auc, best_theta, no_improve = val_auc, self.theta.copy(), 0
                else:
                    no_improve += 1
                if verbose:
                    print(f"  [VQC] epoch {epoch:2d}  val_auc={val_auc:.4f}")
                if no_improve >= patience:
                    break
        self.theta = best_theta
        return best_val_auc
